# 8. Menzerath-Altmann Law (MAL) Analysis

**Summary**: Comprehensive analysis of Menzerath-Altmann Law across 200+ languages from Universal Dependencies, examining how constituent size decreases as the number of dependents increases.

---

## Theoretical Background

**Menzerath-Altmann Law (MAL)** states that the larger a linguistic construct, the smaller its constituents tend to be. In verb-centered analysis:
- As the number of verbal dependents (n) increases, the average constituent size should decrease
- A language with high MAL effect score shows a clear decreasing trend in constituent size as n grows

---

## Notebook Structure

| Section | Contents |
|---------|----------|
| **1. Setup and Data Loading** | Import libraries, load metadata and position statistics |
| **2. Core MAL Computation** | Compute MAL_n scores (total dependents), compliance metrics, heatmaps, HTML report |
| **3. MAL Dynamics Analysis** | Step-by-step compliance, heatmaps |
| **4. Language Family Analysis** | Family-level aggregation, statistical significance tests |
| **5. Typological Analysis** | MAL asymmetry, decay rates, trajectory clustering |
| **6. Summary and Export** | Summary statistics, universality tests, data export |
| **7. Generate UD Language Maps** | Interactive maps of UD languages |

---

## Key Measures Computed

| Measure | Description |
|---------|-------------|
| **MAL_n** | Average constituent size with exactly n total dependents |
| **MAL_left_n / MAL_right_n** | Directional MAL scores (n deps on one side, any on other) |
| **MAL effect score** | Negative slope of MAL_n ~ n (higher = stronger MAL effect) |
| **MAL Asymmetry** | Difference between right and left MAL effect score |
| **Decay Rate** | How quickly constituent size drops (early vs late) |
| **Trajectory Cluster** | Grouping of languages by MAL curve shape |

---

## Prerequisites

| Notebook | Required Data | Purpose |
|----------|--------------|--------|
| `01_data_preparation_and_validation.ipynb` | `metadata.pkl` | Language names, groups, colors |
| `04_data_processing.ipynb` | `all_langs_position2sizes.pkl`, `all_langs_position2num.pkl` | Raw position statistics |
| (Optional) `05_comparative_visualization.ipynb` | `vo_vs_hi_scores.csv` | VO scores for correlation |

---

## Inputs / Outputs

**Inputs**:
- `data/metadata.pkl`
- `data/all_langs_position2sizes.pkl`, `data/all_langs_position2num.pkl`
- `data/vo_vs_hi_scores.csv` (for VO correlation)

**Outputs**:
- `data/lang2MAL_full.pkl` - Full MAL data per language
- `data/mal_compliance_scores.csv` - MAL effect scores
- `data/mal_asymmetry.csv` - Directional asymmetry scores
- `data/mal_decay_rates.csv` - Decay pattern analysis
- `data/mal_trajectories.csv` - Trajectory cluster assignments
- `plots/mal_*.png` - Visualizations

**Runtime**: ~2-3 minutes

---

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy import stats as scipy_stats  # Alias to avoid shadowing by dict later
from collections import defaultdict
from importlib import reload

# Custom modules
import data_utils
import plotting

reload(plotting)

%matplotlib inline

# Configuration
DATA_DIR = "data"
PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Minimum sample count threshold for statistical reliability
# This value is used throughout the notebook and in the HTML report module
MIN_COUNT = 100

## 1. Setup and Data Loading

In [ ]:
# Load metadata
metadata = data_utils.load_metadata(os.path.join(DATA_DIR, 'metadata.pkl'))
langNames = metadata['langNames']
langnameGroup = metadata['langnameGroup']
group_to_color = metadata['appearance_dict']

print(f"Loaded metadata for {len(langNames)} languages")

# Load position statistics (raw counts and sizes)
with open(os.path.join(DATA_DIR, 'all_langs_position2sizes.pkl'), 'rb') as f:
    all_langs_position2sizes = pickle.load(f)

with open(os.path.join(DATA_DIR, 'all_langs_position2num.pkl'), 'rb') as f:
    all_langs_position2num = pickle.load(f)

print(f"Loaded position statistics for {len(all_langs_position2sizes)} languages")

# Load VO scores for correlation analysis
vo_path = os.path.join(DATA_DIR, 'vo_vs_hi_scores.csv')
if os.path.exists(vo_path):
    vo_hi_df = pd.read_csv(vo_path)
    print(f"Loaded VO/HI scores for {len(vo_hi_df)} languages")
else:
    print("Warning: vo_vs_hi_scores.csv not found. Run notebook 04 first.")
    vo_hi_df = pd.DataFrame()

## 2. Core MAL Computation (Total Dependents)

This section computes the primary MAL measure based on **total** number of dependents.

### 2.1 MAL_n Scores

**MAL_n** = Average constituent size when a verb has **total n dependents** (left + right combined = n).

For n=4, this includes ALL configurations:
- VXXXX (4 right, 0 left) → `bilateral_L0_R4_*`
- XVXXX (1 left, 3 right) → `bilateral_L1_R3_*`  
- XXVXX (2 left, 2 right) → `bilateral_L2_R2_*`
- XXXVX (3 left, 1 right) → `bilateral_L3_R1_*`
- XXXXV (4 left, 0 right) → `bilateral_L4_R0_*`

**Note**: Uses bilateral keys in the data. Also computes directional MAL (MAL_right_n, MAL_left_n) for use in Section 3.

In [ ]:
def compute_mal_scores(all_langs_position2sizes, all_langs_position2num, min_count=None):
    """
    Compute Menzerath-Altmann Law scores for all languages.
    
    Computes three types of MAL scores:
    1. MAL_n (total): Average size when verb has TOTAL n dependents (using bilateral keys)
    2. MAL_right_n: Average size when verb has n RIGHT dependents (any left count)
    3. MAL_left_n: Average size when verb has n LEFT dependents (any right count)
    
    NOTE: This function now computes ALL MAL values regardless of count.
    The MIN_COUNT filtering is done at display time in the HTML report,
    where low-count values are shown in grey and excluded from regressions.
    
    Parameters
    ----------
    all_langs_position2sizes : dict
        {lang: {position_key: total_size}}
    all_langs_position2num : dict
        {lang: {position_key: count}}
    min_count : int, optional
        DEPRECATED - kept for backwards compatibility but no longer used.
        Filtering happens at display time.
    
    Returns
    -------
    dict
        {lang: {
            'total': {n: MAL_n},           # Total dependents (all bilateral configs)
            'right': {n: MAL_right_n},     # Right dependents (any left)
            'left': {n: MAL_left_n}        # Left dependents (any right)
        }}
    """
    import re
    
    lang2MAL = {}
    
    # Check if bilateral keys exist in the data
    sample_lang = next(iter(all_langs_position2sizes))
    has_bilateral = any(k.startswith('bilateral_') for k in all_langs_position2sizes[sample_lang])
    if has_bilateral:
        print("✓ Bilateral keys found - computing full MAL_n for all configurations")
    else:
        print("⚠ Bilateral keys not found - using unilateral configs only. Re-run data extraction to get bilateral keys.")
    
    for lang in all_langs_position2sizes:
        # =================================================================
        # 1. MAL_n (TOTAL dependents) - uses BILATERAL keys for all configs
        # =================================================================
        # Bilateral keys: bilateral_L{left}_R{right}_pos_{i}_{side}
        # For total n, we need all (L, R) pairs where L + R = n
        total_n_to_constituents = defaultdict(list)
        
        if has_bilateral:
            # Use bilateral keys for complete MAL_n
            # NO MIN_COUNT FILTERING - compute all values
            for key, size_sum in all_langs_position2sizes[lang].items():
                count = all_langs_position2num[lang].get(key, 0)
                if count == 0:
                    continue
                
                match_bilateral = re.match(r'bilateral_L(\d+)_R(\d+)_pos_(\d+)_(left|right)$', key)
                if match_bilateral:
                    n_left = int(match_bilateral.group(1))
                    n_right = int(match_bilateral.group(2))
                    pos = int(match_bilateral.group(3))
                    side = match_bilateral.group(4)
                    total_n = n_left + n_right
                    avg_size = size_sum / count
                    total_n_to_constituents[total_n].append((avg_size, count))
        else:
            # Fallback: use exact unilateral keys only
            for key, size_sum in all_langs_position2sizes[lang].items():
                count = all_langs_position2num[lang].get(key, 0)
                if count == 0:
                    continue
                
                # Exact right-only: right_i_totright_n (0 left dependents)
                match_exact_right = re.match(r'right_(\d+)_totright_(\d+)$', key)
                if match_exact_right:
                    pos = int(match_exact_right.group(1))
                    tot = int(match_exact_right.group(2))
                    avg_size = size_sum / count
                    total_n_to_constituents[tot].append((avg_size, count))
                    continue
                
                # Exact left-only: left_i_totleft_n (0 right dependents)
                match_exact_left = re.match(r'left_(\d+)_totleft_(\d+)$', key)
                if match_exact_left:
                    pos = int(match_exact_left.group(1))
                    tot = int(match_exact_left.group(2))
                    avg_size = size_sum / count
                    total_n_to_constituents[tot].append((avg_size, count))
                    continue
            
            # Add XVX (bilateral with 1 left, 1 right -> total = 2)
            xvx_keys = ['xvx_left_1', 'xvx_right_1']
            for xvx_key in xvx_keys:
                if xvx_key in all_langs_position2sizes[lang]:
                    size_sum = all_langs_position2sizes[lang][xvx_key]
                    count = all_langs_position2num[lang].get(xvx_key, 0)
                    if count > 0:
                        avg_size = size_sum / count
                        total_n_to_constituents[2].append((avg_size, count))
        
        # =================================================================
        # 2. MAL_right_n (n right deps, ANY left) - uses _anyother_ keys
        # =================================================================
        right_n_to_constituents = defaultdict(list)
        
        for key, size_sum in all_langs_position2sizes[lang].items():
            count = all_langs_position2num[lang].get(key, 0)
            if count == 0:
                continue
            
            match_anyother_right = re.match(r'right_(\d+)_anyother_totright_(\d+)$', key)
            if match_anyother_right:
                pos = int(match_anyother_right.group(1))
                tot = int(match_anyother_right.group(2))
                avg_size = size_sum / count
                right_n_to_constituents[tot].append((avg_size, count))
        
        # =================================================================
        # 3. MAL_left_n (n left deps, ANY right) - uses _anyother_ keys
        # =================================================================
        left_n_to_constituents = defaultdict(list)
        
        for key, size_sum in all_langs_position2sizes[lang].items():
            count = all_langs_position2num[lang].get(key, 0)
            if count == 0:
                continue
            
            match_anyother_left = re.match(r'left_(\d+)_anyother_totleft_(\d+)$', key)
            if match_anyother_left:
                pos = int(match_anyother_left.group(1))
                tot = int(match_anyother_left.group(2))
                avg_size = size_sum / count
                left_n_to_constituents[tot].append((avg_size, count))
        
        # =================================================================
        # Compute arithmetic means (weighted average of sizes)
        # =================================================================
        def compute_mal_from_constituents(n_to_constituents):
            mal_n = {}
            for n, constituents in n_to_constituents.items():
                if not constituents:
                    continue
                total_weighted_size = sum(size * cnt for size, cnt in constituents)
                total_count = sum(cnt for _, cnt in constituents)
                if total_count > 0:
                    mal_n[n] = total_weighted_size / total_count
            return mal_n
        
        lang2MAL[lang] = {
            'total': compute_mal_from_constituents(total_n_to_constituents),
            'right': compute_mal_from_constituents(right_n_to_constituents),
            'left': compute_mal_from_constituents(left_n_to_constituents)
        }
    
    return lang2MAL


# Compute MAL scores (no filtering - all n values computed)
# MIN_COUNT filtering is applied at display time in the HTML report
lang2MAL = compute_mal_scores(all_langs_position2sizes, all_langs_position2num)

# Show summary
print(f"\nComputed MAL scores for {len(lang2MAL)} languages")

# Count languages with data
langs_with_total = sum(1 for l in lang2MAL if lang2MAL[l]['total'])
langs_with_right = sum(1 for l in lang2MAL if lang2MAL[l]['right'])
langs_with_left = sum(1 for l in lang2MAL if lang2MAL[l]['left'])
print(f"  - With MAL_n (total): {langs_with_total}")
print(f"  - With MAL_right_n: {langs_with_right}")
print(f"  - With MAL_left_n: {langs_with_left}")

# Example: show MAL for a sample language
sample_langs = [l for l in lang2MAL.keys() if lang2MAL[l]['total'] and len(lang2MAL[l]['total']) >= 3][:2]
for sample_lang in sample_langs:
    print(f"\n{'='*60}")
    print(f"MAL scores for {langNames.get(sample_lang, sample_lang)}:")
    print(f"{'='*60}")
    
    print("\n  MAL_n (total dependents):")
    for n in sorted(lang2MAL[sample_lang]['total'].keys()):
        print(f"    n={n}: MAL = {lang2MAL[sample_lang]['total'][n]:.3f}")
    
    print("\n  MAL_right_n (n right deps, any left):")
    for n in sorted(lang2MAL[sample_lang]['right'].keys()):
        print(f"    n={n}: MAL = {lang2MAL[sample_lang]['right'][n]:.3f}")
    
    print("\n  MAL_left_n (n left deps, any right):")
    for n in sorted(lang2MAL[sample_lang]['left'].keys()):
        print(f"    n={n}: MAL = {lang2MAL[sample_lang]['left'][n]:.3f}")

### 2.2 MAL Effect Scores

The **MAL Effect Score** quantifies how well a language follows the Menzerath-Altmann Law.

#### Intuition

If MAL holds, then as n (number of dependents) increases, the average constituent size (MAL_n) should **decrease**. A language with high MAL effect score will show:
- MAL_1 > MAL_2 > MAL_3 > MAL_4 (decreasing trend)

#### How It's Computed

We fit a linear regression: $\text{MAL}_n = \alpha + \beta \cdot n$

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Slope (β)** | From regression | Negative = sizes decrease with n (MAL holds) |
| **Normalized Slope** | $\beta / \text{MAL}_1$ | Scale-independent version |
| **MAL effect score** | $-\beta / \text{MAL}_1$ | **Higher = stronger MAL effect** |
| **Spearman ρ** | Rank correlation of n vs MAL_n | More negative = stronger monotonic decrease |
| **Decrease Ratio** | $\text{MAL}_1 / \text{MAL}_{\max}$ | > 1 means sizes decreased overall |

### Example

If a language has MAL_1=3.5, MAL_2=2.8, MAL_3=2.2, MAL_4=1.9:
- Slope β ≈ -0.52 (negative → good)
- Normalized slope ≈ -0.15
- **MAL effect score ≈ 0.15** (positive → follows MAL)
- Spearman ρ = -1.0 (perfect monotonic decrease)

In [ ]:
def compute_loglog_regression(mal_data, start_n=1):
    """
    Compute linear regression in log-log space: log(MAL_n) = a + slope * log(n)
    
    The negative of the slope (-slope) is the MAL effect score (β).
    Under MAL: log(MAL_n) = a - β * log(n), so β = -slope.
    
    Args:
        mal_data: Dict mapping n -> MAL value
        start_n: Starting n value for regression (1 for full, 3 for excluding small n)
        
    Returns:
        Dict with 'slope', 'intercept', 'r_squared', 'beta' (MAL effect score = -slope),
        or None if insufficient data
    """
    # Filter data points with n >= start_n
    points = [(n, mal) for n, mal in mal_data.items() if n >= start_n and mal > 0]
    
    if len(points) < 2:
        return None
    
    # Sort by n
    points.sort(key=lambda x: x[0])
    
    # Convert to log space
    log_n = np.array([np.log(p[0]) for p in points])
    log_mal = np.array([np.log(p[1]) for p in points])
    
    # Linear regression: log_mal = intercept + slope * log_n
    try:
        slope, intercept = np.polyfit(log_n, log_mal, 1)
        
        # Compute R-squared
        predicted = intercept + slope * log_n
        ss_res = np.sum((log_mal - predicted) ** 2)
        ss_tot = np.sum((log_mal - np.mean(log_mal)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        return {
            'slope': float(slope),
            'intercept': float(intercept),
            'r_squared': float(r_squared),
            'beta': float(-slope),  # MAL effect score = negative of slope
            'n_points': len(points),
            'n_range': (points[0][0], points[-1][0])
        }
    except Exception:
        return None


def compute_mal_compliance(lang2MAL, min_n_values=3, score_type='total'):
    """
    Compute MAL effect scores for each language, including log-log regression β scores.
    
    Parameters
    ----------
    lang2MAL : dict
        {lang: {'total': {n: MAL_n}, 'right': {...}, 'left': {...}}}
    min_n_values : int
        Minimum number of n values required to compute compliance
    score_type : str
        Which MAL type to use ('total', 'right', or 'left')
    
    Returns
    -------
    pd.DataFrame
        DataFrame with compliance metrics per language, including:
        - beta_1max: MAL effect from log-log regression (n=1 to max)
        - beta_3max: MAL effect from log-log regression (n=3 to max)
        - r2_1max, r2_3max: R² values for the regressions
    """
    results = []
    
    for lang, mal_data_all in lang2MAL.items():
        mal_data = mal_data_all.get(score_type, {})
        
        if len(mal_data) < min_n_values:
            continue
        
        # Get n values and corresponding MAL values
        ns = sorted(mal_data.keys())
        mals = [mal_data[n] for n in ns]
        
        # 1. Linear regression slope (original)
        slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(ns, mals)
        
        # 2. Spearman correlation (more robust to outliers)
        spearman_r, spearman_p = stats.spearmanr(ns, mals)
        
        # 3. Decrease ratio: MAL_1 / MAL_max (if MAL decreases, this should be > 1)
        mal_first = mal_data.get(min(ns), np.nan)
        mal_last = mal_data.get(max(ns), np.nan)
        decrease_ratio = mal_first / mal_last if mal_last > 0 else np.nan
        
        # 4. Normalized slope (slope divided by MAL_1 to account for scale)
        normalized_slope = slope / mal_first if mal_first > 0 else np.nan
        
        # 5. Original Compliance score: negative of normalized slope (positive = good MAL)
        mal_compliance = -normalized_slope
        
        # 6. NEW: Log-log regression β scores
        reg_1max = compute_loglog_regression(mal_data, start_n=1)
        reg_2max = compute_loglog_regression(mal_data, start_n=2)
        reg_3max = compute_loglog_regression(mal_data, start_n=3)
        
        beta_1max = reg_1max['beta'] if reg_1max else np.nan
        beta_2max = reg_2max['beta'] if reg_2max else np.nan
        beta_3max = reg_3max['beta'] if reg_3max else np.nan
        r2_1max = reg_1max['r_squared'] if reg_1max else np.nan
        r2_2max = reg_2max['r_squared'] if reg_2max else np.nan
        r2_3max = reg_3max['r_squared'] if reg_3max else np.nan
        
        results.append({
            'language_code': lang,
            'score_type': score_type,
            'n_values': len(ns),
            'max_n': max(ns),
            'MAL_1': mal_first,
            'MAL_max': mal_last,
            'slope': slope,
            'normalized_slope': normalized_slope,
            'spearman_r': spearman_r,
            'spearman_p': spearman_p,
            'r_squared': r_value ** 2,
            'decrease_ratio': decrease_ratio,
            'mal_compliance': mal_compliance,
            # New β scores from log-log regression
            'beta_1max': beta_1max,
            'beta_2max': beta_2max,
            'beta_3max': beta_3max,
            'r2_1max': r2_1max,
            'r2_2max': r2_2max,
            'r2_3max': r2_3max,
        })
    
    return pd.DataFrame(results)


# Compute compliance scores for all three types
mal_compliance_total = compute_mal_compliance(lang2MAL, min_n_values=3, score_type='total')
mal_compliance_right = compute_mal_compliance(lang2MAL, min_n_values=3, score_type='right')
mal_compliance_left = compute_mal_compliance(lang2MAL, min_n_values=3, score_type='left')

# Use TOTAL as the main compliance dataframe (primary MAL_n)
mal_compliance_df = mal_compliance_total.copy()

# Add language names and groups to all dataframes
for df in [mal_compliance_df, mal_compliance_right, mal_compliance_left]:
    if len(df) > 0:
        df['language_name'] = df['language_code'].map(lambda x: langNames.get(x, x))
        df['group'] = df['language_name'].map(lambda x: langnameGroup.get(x, 'Other'))

print(f"{'='*70}")
print(f"MAL EFFECT SCORES COMPUTED")
print(f"{'='*70}")
print(f"MAL_n (total dependents): {len(mal_compliance_total)} languages")
print(f"MAL_right_n (directional): {len(mal_compliance_right)} languages")
print(f"MAL_left_n (directional): {len(mal_compliance_left)} languages")

if len(mal_compliance_df) > 0:
    print(f"\n{'='*70}")
    print(f"NEW: Log-Log Regression β Scores (MAL Effect)")
    print(f"{'='*70}")
    print(f"β > 0 means MAL conformant (constituent size decreases as n increases)")
    print(f"β_1max: Regression from n=1 to max")
    print(f"β_3max: Regression from n=3 to max (excludes potentially atypical n=1,2)")
    
    print(f"\n{'='*70}")
    print(f"Top 10 languages by β_1max (strongest MAL effect):")
    print(f"{'='*70}")
    top10 = mal_compliance_df.nlargest(10, 'beta_1max')[['language_name', 'group', 'beta_1max', 'r2_1max', 'beta_3max', 'r2_3max']]
    print(top10.to_string(index=False))
    
    print(f"\n{'='*70}")
    print(f"Bottom 10 languages by β_1max (weakest/anti-MAL):")
    print(f"{'='*70}")
    bottom10 = mal_compliance_df.nsmallest(10, 'beta_1max')[['language_name', 'group', 'beta_1max', 'r2_1max', 'beta_3max', 'r2_3max']]
    print(bottom10.to_string(index=False))
    
    # Summary statistics
    print(f"\n{'='*70}")
    print(f"β Score Summary Statistics")
    print(f"{'='*70}")
    for col in ['beta_1max', 'beta_3max']:
        valid = mal_compliance_df[col].dropna()
        print(f"\n{col}:")
        print(f"  Mean: {valid.mean():.4f}, Std: {valid.std():.4f}")
        print(f"  MAL-conformant (β > 0): {(valid > 0).sum()}/{len(valid)} ({100*(valid > 0).mean():.1f}%)")
        print(f"  Strong MAL (β > 0.1): {(valid > 0.1).sum()}/{len(valid)} ({100*(valid > 0.1).mean():.1f}%)")
else:
    print("\nNote: Not enough languages with 3+ total-MAL values. Using directional scores.")
    # Fallback to right-side scores if total is insufficient
    if len(mal_compliance_right) > len(mal_compliance_df):
        mal_compliance_df = mal_compliance_right.copy()
        print(f"Using MAL_right_n as primary score ({len(mal_compliance_df)} languages)")

### 2.3 MAL_n Curves Visualization

In [ ]:
# Aggregate MAL effect score by language family - including new β scores
family_mal = mal_compliance_df.groupby('group').agg({
    'mal_compliance': ['mean', 'std', 'count'],
    'beta_1max': ['mean', 'std'],
    'beta_2max': ['mean', 'std'],
    'beta_3max': ['mean', 'std'],
    'spearman_r': 'mean',
    'slope': 'mean',
    'decrease_ratio': 'mean'
}).round(4)

family_mal.columns = ['_'.join(col).strip() for col in family_mal.columns.values]
family_mal = family_mal.sort_values('beta_1max_mean', ascending=False)

print("MAL effect score by Language Family (sorted by β_1max):")
print(family_mal[['mal_compliance_count', 'beta_1max_mean', 'beta_1max_std', 'beta_2max_mean', 'beta_2max_std', 'beta_3max_mean', 'beta_3max_std']].to_string())

# Global average
print(f"\n{'='*60}")
print(f"Global MAL Effect Scores (β from log-log regression):")
print(f"{'='*60}")
print(f"β_1max (n=1→max):")
print(f"  Mean: {mal_compliance_df['beta_1max'].mean():.4f}")
print(f"  Std:  {mal_compliance_df['beta_1max'].std():.4f}")
print(f"β_2max (n=2→max):")
print(f"  Mean: {mal_compliance_df['beta_2max'].mean():.4f}")
print(f"  Std:  {mal_compliance_df['beta_2max'].std():.4f}")
print(f"β_3max (n=3→max):")
print(f"  Mean: {mal_compliance_df['beta_3max'].mean():.4f}")
print(f"  Std:  {mal_compliance_df['beta_3max'].std():.4f}")
print(f"\nMean Spearman r: {mal_compliance_df['spearman_r'].mean():.4f}")

### 2.4 VO Score Correlation

Investigate relationships between MAL effect score and word order typology.

In [ ]:
# Merge MAL effect score with VO scores
if not vo_hi_df.empty:
    print("Available columns in vo_hi_df:", vo_hi_df.columns.tolist())
    
    join_col = 'language_code' if 'language_code' in vo_hi_df.columns else 'language'
    
    # Build list of columns to merge (only include existing columns)
    merge_cols = [join_col]
    if 'vo_score' in vo_hi_df.columns:
        merge_cols.append('vo_score')
    if 'head_initiality' in vo_hi_df.columns:
        merge_cols.append('head_initiality')
    elif 'head_initiality_score' in vo_hi_df.columns:
        merge_cols.append('head_initiality_score')
    
    if len(merge_cols) > 1:  # Have at least one score column
        mal_vo_df = pd.merge(
            mal_compliance_df,
            vo_hi_df[merge_cols].rename(columns={join_col: 'language_code', 
                                                  'head_initiality_score': 'head_initiality'}),
            on='language_code',
            how='inner'
        )
        
        print(f"Merged data for {len(mal_vo_df)} languages")
        
        # Compute correlations for β scores with VO score
        if 'vo_score' in mal_vo_df.columns:
            print(f"\n{'='*60}")
            print("Correlations with VO Score:")
            print(f"{'='*60}")
            corr_beta_1max = mal_vo_df['beta_1max'].corr(mal_vo_df['vo_score'])
            corr_beta_3max = mal_vo_df['beta_3max'].corr(mal_vo_df['vo_score'])
            corr_old = mal_vo_df['mal_compliance'].corr(mal_vo_df['vo_score'])
            print(f"  β_1max ~ VO Score: r = {corr_beta_1max:.4f}")
            print(f"  β_3max ~ VO Score: r = {corr_beta_3max:.4f}")
            print(f"  (old mal_compliance ~ VO Score: r = {corr_old:.4f})")
        
        if 'head_initiality' in mal_vo_df.columns:
            print(f"\n{'='*60}")
            print("Correlations with Head-Initiality:")
            print(f"{'='*60}")
            corr_hi_1max = mal_vo_df['beta_1max'].corr(mal_vo_df['head_initiality'])
            corr_hi_3max = mal_vo_df['beta_3max'].corr(mal_vo_df['head_initiality'])
            print(f"  β_1max ~ Head-Initiality: r = {corr_hi_1max:.4f}")
            print(f"  β_3max ~ Head-Initiality: r = {corr_hi_3max:.4f}")
    else:
        mal_vo_df = mal_compliance_df.copy()
        print("No score columns (vo_score, head_initiality) found in vo_hi_df")
else:
    mal_vo_df = mal_compliance_df.copy()
    print("No VO data available for correlation analysis")

(Visualization cells follow - MAL_n curves are part of Section 2.3)

#### MAL_n Curves (Total Dependents)

In [ ]:
# Plot MAL_n curves (total dependents)
lang2MAL_total = {lang: data['total'] for lang, data in lang2MAL.items() if data['total']}

fig, ax = plt.subplots(figsize=(14, 8))

# Filter languages with data from n=1 to 4
ns = list(range(1, 5))
filtered_data = {
    lang: data for lang, data in lang2MAL_total.items()
    if all(n in data for n in ns)
}
print(f"Plotting {len(filtered_data)} languages with MAL_n data for n=1 to 4")

if filtered_data:
    # Plot individual curves
    for lang, data in filtered_data.items():
        lang_name = langNames.get(lang, lang)
        group = langnameGroup.get(lang_name, 'Other')
        color = group_to_color.get(group, 'gray')
        values = [data[n] for n in ns]
        ax.plot(ns, values, alpha=0.3, color=color)
    
    # Plot mean curve
    mean_vals = [np.mean([filtered_data[l][n] for l in filtered_data]) for n in ns]
    ax.plot(ns, mean_vals, color='black', linewidth=2.5, label=f'Mean (n={len(filtered_data)})')
    print(f"Mean MAL values: {[f'{v:.3f}' for v in mean_vals]}")
    
    ax.set_xlabel("Total number of dependents (n)", fontsize=12)
    ax.set_ylabel("Average constituent size (MAL_n)", fontsize=12)
    ax.set_title(f"MAL_n: Constituent Size vs Total Dependents ({len(filtered_data)} languages)")
    ax.set_xticks(ns)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_n_total_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_n_total_curves.png")
else:
    print("Not enough languages with complete MAL_n data for n=1 to 4")
    plt.close()

In [ ]:
# Plot MAL_n curves (total dependents)
lang2MAL_total = {lang: data['total'] for lang, data in lang2MAL.items() if data['total']}

fig, ax = plt.subplots(figsize=(14, 8))

# Find the maximum n value available across all languages (capped at 6)
all_n_values = set()
for data in lang2MAL_total.values():
    all_n_values.update(data.keys())
max_n = min(max(all_n_values) if all_n_values else 4, 6)  # Cap at 6
ns = list(range(1, max_n + 1))

# Filter languages that have at least n=1 data
filtered_data = {
    lang: data for lang, data in lang2MAL_total.items()
    if 1 in data  # Only require n=1
}
print(f"Plotting {len(filtered_data)} languages with MAL_n data (n=1 to {max_n}, variable length)")

# Use global MIN_COUNT defined at the top of the notebook

if filtered_data:
    # Plot individual curves (lines extend only as far as data is available)
    for lang, data in filtered_data.items():
        lang_name = langNames.get(lang, lang)
        group = langnameGroup.get(lang_name, 'Other')
        color = group_to_color.get(group, 'gray')
        # Get available n values for this language
        lang_ns = sorted([n for n in ns if n in data])
        values = [data[n] for n in lang_ns]
        ax.plot(lang_ns, values, alpha=0.3, color=color)
    
    # Plot mean curve (only where we have data)
    mean_vals = []
    mean_ns = []
    for n in ns:
        vals = [filtered_data[l][n] for l in filtered_data if n in filtered_data[l]]
        if vals:  # Only add if we have data for this n
            mean_vals.append(np.mean(vals))
            mean_ns.append(n)
    
    ax.plot(mean_ns, mean_vals, color='black', linewidth=2.5, label=f'Mean (n={len(filtered_data)})')
    print(f"Mean MAL values: {[f'n={n}: {v:.3f} (from {sum(1 for l in filtered_data if n in filtered_data[l])} langs)' for n, v in zip(mean_ns, mean_vals)]}")
    
    ax.set_xlabel("Total number of dependents (n)", fontsize=12)
    ax.set_ylabel("Average constituent size (MAL_n)", fontsize=12)
    ax.set_title(f"MAL_n: Constituent Size vs Total Dependents ({len(filtered_data)} languages)\n(minimum {MIN_COUNT} occurrences per configuration)", fontsize=12)
    ax.set_xticks(ns)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_n_total_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_n_total_curves.png")
else:
    print("Not enough languages with MAL_n data")
    plt.close()

#### Interpretation: MAL_n Curves

The plot above shows the **Menzerath-Altmann Law (MAL)** across languages. Each line represents a language, with the x-axis showing the total number of dependents (n) and the y-axis showing the average constituent size (MAL_n).

**Key observations:**

1. **Strong MAL effect for n=1 to 3**: The mean curve (black line) shows a clear downward trend from n=1 to n=3, confirming MAL — as the number of dependents increases, the average size of each constituent decreases.

2. **Upturn at higher n values**: The mean curve rises from n=3 to n=6. This is likely a **sampling artifact**: at higher n values, only languages with sufficient data (≥10 occurrences) contribute to the mean. These tend to be languages with larger corpora or more complex verbal constructions, which may have systematically different properties than the full set of languages.

3. **Survivor bias**: As n increases, fewer languages have enough data to be included. The languages that "survive" to n=5 or n=6 are not a random sample — they may represent specific typological profiles or simply better-resourced languages.

4. **Cross-linguistic pattern (n=1 to 3)**: The consistent downward pattern across diverse language families (indicated by colors) for low n values suggests MAL is a linguistic universal, driven by cognitive processing constraints and communicative efficiency.

5. **Variation around the mean**: The spread of individual language curves indicates cross-linguistic variation in the strength of the MAL effect, which may correlate with typological features such as head-directionality (VO vs OV).

### 2.5 MAL vs VO Scatter Plot

In [ ]:
# MAL Effect (β scores) vs VO Score - Two plots for β_1max and β_3max
# This takes ~40 seconds per plot

if 'vo_score' in mal_vo_df.columns:
    # Plot 1: β_1max vs VO Score
    plotting.plot_scatter_2d(
        df=mal_vo_df,
        x_col='vo_score',
        y_col='beta_1max',
        group_col='group',
        appearance_dict=group_to_color,
        title='MAL Effect (β from n=1→max) vs VO Score',
        xlabel='VO Score (higher = more VO)',
        ylabel='β_1max (MAL effect: positive = constituent shrinkage)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_beta_1max_vs_vo_score.png'
    )
    
    # Plot 2: β_3max vs VO Score
    plotting.plot_scatter_2d(
        df=mal_vo_df.dropna(subset=['beta_3max']),
        x_col='vo_score',
        y_col='beta_3max',
        group_col='group',
        appearance_dict=group_to_color,
        title='MAL Effect (β from n=3→max) vs VO Score',
        xlabel='VO Score (higher = more VO)',
        ylabel='β_3max (MAL effect: positive = constituent shrinkage)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_beta_3max_vs_vo_score.png'
    )
else:
    print("No VO score data available for scatter plot")

### 2.6 Spearman vs VO Scatter

In [ ]:
# R² of log-log regression vs VO Score - shows how well the power law fits
# This takes ~50 seconds per plot

if 'vo_score' in mal_vo_df.columns:
    # R² for β_1max regression
    plotting.plot_scatter_2d(
        df=mal_vo_df,
        x_col='vo_score',
        y_col='r2_1max',
        group_col='group',
        appearance_dict=group_to_color,
        title='Log-Log Regression R² (n=1→max) vs VO Score',
        xlabel='VO Score (higher = more VO)',
        ylabel='R² (goodness of fit for log(MAL) ~ log(n))',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_r2_1max_vs_vo_score.png'
    )
    
    # Spearman r as alternative metric
    plotting.plot_scatter_2d(
        df=mal_vo_df,
        x_col='vo_score',
        y_col='spearman_r',
        group_col='group',
        appearance_dict=group_to_color,
        title='MAL Spearman Correlation vs VO Score',
        xlabel='VO Score (higher = more VO)',
        ylabel='Spearman r (negative = stronger MAL)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_spearman_vs_vo_score.png'
    )
else:
    print("No VO score data available for scatter plot")

### 2.7 MAL effect score vs Spearman Scatter
Plot the relationship between MAL effect scores and Spearman correlation coefficients across languages.


In [ ]:
# Scatter plots: β scores vs Spearman correlation - compare metrics
# this takes 1 min 18 seconds
if 'beta_1max' in mal_vo_df.columns and 'spearman_r' in mal_vo_df.columns:
    # β_1max vs Spearman r
    plotting.plot_scatter_2d(
        df=mal_vo_df,
        x_col='spearman_r',
        y_col='beta_1max',
        group_col='group',
        appearance_dict=group_to_color,
        title='β_1max vs Spearman Correlation',
        xlabel='Spearman r (negative = stronger MAL)',
        ylabel='β_1max (MAL effect from log-log regression)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_beta_1max_vs_spearman.png'
    )
    
    # β_3max vs Spearman r
    plotting.plot_scatter_2d(
        df=mal_vo_df.dropna(subset=['beta_3max']),
        x_col='spearman_r',
        y_col='beta_3max',
        group_col='group',
        appearance_dict=group_to_color,
        title='β_3max vs Spearman Correlation',
        xlabel='Spearman r (negative = stronger MAL)',
        ylabel='β_3max (MAL effect from n=3→max)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_beta_3max_vs_spearman.png'
    )
else:
    print("Required columns not available for scatter plot")

In [ ]:
# Scatter plot: β_1max vs β_2max - compare MAL effect with/without n=1
# This shows whether including n=1 changes the MAL effect score

if 'beta_1max' in mal_vo_df.columns and 'beta_2max' in mal_vo_df.columns:
    valid_df = mal_vo_df.dropna(subset=['beta_1max', 'beta_2max'])
    
    # Compute correlation
    corr = valid_df['beta_1max'].corr(valid_df['beta_2max'])
    print(f"Correlation between β_1max and β_2max: r = {corr:.4f}")
    print(f"Languages with both scores: {len(valid_df)}")
    
    plotting.plot_scatter_2d(
        df=valid_df,
        x_col='beta_1max',
        y_col='beta_2max',
        group_col='group',
        appearance_dict=group_to_color,
        title=f'β₁₋ₘₐₓ vs β₂₋ₘₐₓ (r = {corr:.3f})',
        xlabel='β₁₋ₘₐₓ (MAL effect from n=1→max)',
        ylabel='β₂₋ₘₐₓ (MAL effect from n=2→max)',
        figsize=(14, 10),
        label_col='language_name',
        with_labels=True,
        add_regression=True,
        plots_dir=PLOTS_DIR,
        filename='mal_beta_1max_vs_beta_2max.png'
    )
    
    # Show diagonal reference line (y=x)
    print("Note: Points above the diagonal have stronger MAL effect when n=1 is excluded")
    print("Points below the diagonal have weaker MAL effect when n=1 is excluded")
else:
    print("Required columns beta_1max and/or beta_2max not available")

#### Interpretation: MAL effect score vs Spearman Correlation

**What this plot shows:**
This scatter plot compares two different metrics for measuring MAL adherence:

- **X-axis (Spearman r)**: A non-parametric rank correlation between valency ($n$) and constituent size. More negative values indicate stronger monotonic decrease (stronger MAL).
- **Y-axis (MAL effect score)**: A normalized slope-based metric derived from linear regression of log-transformed MAL curves. Higher values indicate stronger MAL effect.

**Expected relationship:**
Since both metrics measure the same underlying phenomenon (constituent size decreasing with valency), we expect a **strong negative correlation**: languages with more negative Spearman $r$ should have higher MAL effect score (β)s.

**Key observations:**
- A tight linear relationship validates that both metrics capture the same phenomenon consistently
- Outliers may indicate languages where the MAL relationship is non-monotonic or non-linear
- The regression line slope indicates how the two metrics scale relative to each other

---

**Ceiling/Floor Effects and Discriminative Power:**

A notable pattern in this plot is that languages with **extreme Spearman values** (approaching -1 or +1) show considerable **vertical spread** in their MAL effect scores. This phenomenon is known as a **ceiling effect** (or floor effect for -1):

- **Spearman correlation is bounded** between -1 and +1. Once the relationship is perfectly monotonic, the correlation saturates — it cannot distinguish between a gentle monotonic decrease and a steep one.
- **MAL effect score remains unbounded** and captures the actual *magnitude* of the slope, not just its direction or monotonicity.

For example, two languages might both have Spearman $r = -1$ (perfect negative monotonic relationship), but one might show a steep decline in constituent size (high MAL effect score) while another shows only a gradual decline (lower MAL effect score). The Spearman correlation conflates these cases; the MAL effect score score differentiates them.

**Implication for metric choice:**

This suggests that **MAL effect score is the more informative metric** for cross-linguistic comparison, particularly when:
1. Many languages cluster at extreme Spearman values
2. The research question concerns the *strength* of the MAL effect, not just its presence
3. Fine-grained distinctions between languages with similar monotonic patterns are needed

The Spearman correlation remains useful as a **robustness check** (confirming monotonicity) and for cases where the relationship may be non-linear, but the MAL effect score score provides greater **discriminative resolution** across the full range of MAL behavior.

### 2.8 Directional MAL Analysis (Left vs Right)

#### Directional MAL Curves

This section analyzes MAL separately for left-side and right-side dependents.

In [ ]:
# Plot directional MAL curves: MAL_right_n and MAL_left_n
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

titles = {
    'right': 'MAL_right_n (n right deps, any left)',
    'left': 'MAL_left_n (n left deps, any right)'
}

for ax, side in zip(axes, ['left', 'right']):
    lang2MAL_side = {lang: data[side] for lang, data in lang2MAL.items() if data[side]}
    
    # Filter languages with data from n=1 to 4
    ns = list(range(1, 5))
    filtered_data = {
        lang: data for lang, data in lang2MAL_side.items()
        if all(n in data for n in ns)
    }
    
    if filtered_data:
        # Plot individual curves
        for lang, data in filtered_data.items():
            lang_name = langNames.get(lang, lang)
            group = langnameGroup.get(lang_name, 'Other')
            color = group_to_color.get(group, 'gray')
            values = [data[n] for n in ns]
            ax.plot(ns, values, alpha=0.3, color=color)
        
        # Plot mean curve
        mean_vals = [np.mean([filtered_data[l][n] for l in filtered_data]) for n in ns]
        ax.plot(ns, mean_vals, color='black', linewidth=2.5, label=f'Mean (n={len(filtered_data)})')
    
    ax.set_xlabel(f"Number of {side} dependents (n)", fontsize=11)
    ax.set_ylabel("Average constituent size", fontsize=11)
    ax.set_title(titles[side], fontsize=12)
    ax.set_xticks(ns)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_directional_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {PLOTS_DIR}/mal_directional_curves.png")

#### Interpretation: Directional MAL Curves

**What these plots show:**
- **Left panel (MAL_right_n)**: Average constituent size when a head has *n* right-side dependents (with any number of left dependents). A decreasing curve means constituents shrink as more right dependents are added.
- **Right panel (MAL_left_n)**: Average constituent size when a head has *n* left-side dependents (with any number of right dependents).

**Key observations:**
- Both curves typically show **negative slopes** (decreasing size with more dependents), confirming MAL operates on both sides of the head
- The **steepness** of each curve reflects how strongly that side exhibits MAL
- Languages where **right curve is steeper** than left → stronger MAL effect on the right (typical for VO languages)
- Languages where **left curve is steeper** than right → stronger MAL effect on the left (typical for OV languages)
- The **black mean line** shows the cross-linguistic average trend

---

### Additional Directional MAL Measures to Consider

| Measure | Description | Linguistic Interpretation |
|---------|-------------|---------------------------|
| **Directional Slope Ratio** | `slope_right / slope_left` | Values >1 indicate stronger MAL on right side; <1 indicates left-side dominance |
| **Directional R² Comparison** | Compare R² of right vs left linear fits | Which side shows more consistent MAL behavior? |
| **Crossover Point** | At what *n* does left MAL = right MAL? | Identifies asymmetry threshold |
| **Directional MAL Difference** | `MAL_right_n - MAL_left_n` for each *n* | How size differs by direction at each dependent count |
| **Interaction Effect** | Size when n_left × n_right jointly considered | Does having dependents on both sides amplify/dampen MAL? |
| **Positional Decay** | How does MAL vary by *position* within left/right dependents? | First dependent vs. second vs. third on each side |
| **Head-Dependent Asymmetry** | Compare dependent size reduction left vs right of their own heads | Recursive MAL within the dependency tree |
| **Directional Compliance Rate** | % of languages with slope < 0 for each direction | Is MAL more universal on one side? |
| **Variance by Direction** | Std of constituent sizes at each *n* per direction | Which direction has more stable MAL effect? |

### 2.9 MAL HTML Report with Heatmap

In [ ]:
# Generate interactive HTML report for MAL_n analysis
import mal_html_report
from importlib import reload
reload(mal_html_report)  # Reload in case of changes

# Prepare data - now includes ALL n values (no filtering during MAL computation)
lang2MAL_total = {lang: data['total'] for lang, data in lang2MAL.items() if data['total']}
lang2MAL_left = {lang: data['left'] for lang, data in lang2MAL.items() if data.get('left')}
lang2MAL_right = {lang: data['right'] for lang, data in lang2MAL.items() if data.get('right')}

# Get sample counts per n (no filtering - all counts returned)
# MIN_COUNT filtering is now done at display time in the HTML report
lang2counts = mal_html_report.get_sample_counts_per_n(all_langs_position2num)

# Get directional counts for left/right tables (no filtering)
lang2counts_left, lang2counts_right = mal_html_report.get_directional_counts(all_langs_position2num)

# Build lang_to_vo mapping from vo_hi_df if available
lang_to_vo = None
if 'vo_hi_df' in dir() and not vo_hi_df.empty and 'vo_score' in vo_hi_df.columns:
    join_col = 'language_code' if 'language_code' in vo_hi_df.columns else 'language'
    lang_to_vo = {}
    for _, row in vo_hi_df.iterrows():
        lang_code = row[join_col]
        if pd.notna(row['vo_score']):
            lang_to_vo[lang_code] = row['vo_score']
    print(f"✓ VO scores available for {len(lang_to_vo)} languages")

# Path to WALS languages data for geographic coordinates
WALS_LANGUAGES_PATH = os.path.join(DATA_DIR, 'wals', 'languages.csv')

# Generate HTML report with directional sections
HTML_DIR = 'html_analyses'
os.makedirs(HTML_DIR, exist_ok=True)
html_path = os.path.join(HTML_DIR, 'mal_n_analysis.html')

# Generate report - min_count is used for display filtering (grey cells, exclude from regression)
stats = mal_html_report.generate_mal_html_report(
    lang2MAL_total=lang2MAL_total,
    lang2counts=lang2counts,
    langNames=langNames,
    output_path=html_path,
    min_count=MIN_COUNT,  # Used for display: values below this shown in grey
    langnameGroup=langnameGroup,
    wals_languages_path=WALS_LANGUAGES_PATH,
    lang2MAL_left=lang2MAL_left,
    lang2MAL_right=lang2MAL_right,
    lang2counts_left=lang2counts_left,
    lang2counts_right=lang2counts_right,
    lang_to_vo=lang_to_vo
)

print(f"✓ Generated HTML report: {stats['output_path']}")
print(f"  - {stats['n_languages']} languages included")
print(f"  - MAL values for n=1 to {stats['max_n']}")
print(f"  - Local change scores for {stats['n_transitions']} transitions")
if 'n_languages_left' in stats:
    print(f"  - Left dependents: {stats['n_languages_left']} languages with data")
    print(f"  - Right dependents: {stats['n_languages_right']} languages with data")
print(f"\nTransition statistics:")
for trans in sorted(stats['chart_data'].keys()):
    scores = stats['chart_data'][trans]
    print(f"  {trans}: n={len(scores)}, mean={np.mean(scores):.3f}, median={np.median(scores):.3f}")

In [ ]:
# Generate 3 separate HTML reports based on VO score
# 1. VO languages (vo_score > 0.66 = 66%)
# 2. OV languages (vo_score < 0.33 = 33%)
# 3. Mixed languages (0.33 <= vo_score <= 0.66)

from importlib import reload
reload(mal_html_report)

# Load VO scores
vo_path = os.path.join(DATA_DIR, 'vo_vs_hi_scores.csv')
if not os.path.exists(vo_path):
    print("Error: vo_vs_hi_scores.csv not found. Run notebook 04 first.")
else:
    vo_hi_df = pd.read_csv(vo_path)
    print(f"VO/HI data loaded: {len(vo_hi_df)} languages")
    print(f"Columns: {list(vo_hi_df.columns)}")
    
    # Create language code to VO score mapping
    lang_to_vo = {}
    for _, row in vo_hi_df.iterrows():
        lang_code = row.get('language_code')
        if pd.notna(lang_code) and 'vo_score' in row and pd.notna(row['vo_score']):
            lang_to_vo[lang_code] = row['vo_score']
    
    print(f"\nLanguage codes with VO scores: {len(lang_to_vo)}")
    print(f"Sample lang codes: {list(lang_to_vo.keys())[:5]}")
    print(f"Sample MAL lang codes: {list(lang2MAL.keys())[:5]}")
    
    # Filter languages by VO score (note: vo_score is 0-1, not 0-100)
    vo_langs = {lang for lang, score in lang_to_vo.items() if score > 0.66}
    ov_langs = {lang for lang, score in lang_to_vo.items() if score < 0.33}
    mixed_langs = {lang for lang, score in lang_to_vo.items() if 0.33 <= score <= 0.66}
    
    # Check overlap with MAL data
    mal_langs = set(lang2MAL.keys())
    print(f"\nLanguages in MAL data: {len(mal_langs)}")
    print(f"Overlap with VO scores: {len(mal_langs & set(lang_to_vo.keys()))}")
    
    print(f"\n  - VO (>66%): {len(vo_langs)} ({len(vo_langs & mal_langs)} with MAL)")
    print(f"  - OV (<33%): {len(ov_langs)} ({len(ov_langs & mal_langs)} with MAL)")  
    print(f"  - Mixed (33-66%): {len(mixed_langs)} ({len(mixed_langs & mal_langs)} with MAL)")
    
    # Get directional counts (pure left, pure right)
    lang2counts_left, lang2counts_right = mal_html_report.get_directional_counts(
        all_langs_position2num, min_count=MIN_COUNT
    )
    
    # Filter lang2MAL for each subset
    def filter_lang2MAL(lang_set):
        return {lang: data for lang, data in lang2MAL.items() 
                if lang in lang_set and (data.get('total') or data.get('left') or data.get('right'))}
    
    subsets = [
        ('vo_languages', vo_langs, 'VO Languages (VO Score > 66%)', 
         'Head-initial/VO languages where the verb typically precedes its object.'),
        ('ov_languages', ov_langs, 'OV Languages (VO Score < 33%)',
         'Head-final/OV languages where the verb typically follows its object.'),
        ('mixed_languages', mixed_langs, 'Mixed Languages (VO Score 33-66%)',
         'Languages with mixed or flexible word order.')
    ]
    
    for filename, lang_set, title, description in subsets:
        filtered_lang2MAL = filter_lang2MAL(lang_set)
        
        if not filtered_lang2MAL:
            print(f"\n⚠ Skipping {title}: No languages with MAL data")
            continue
        
        # Filter counts for this subset
        filtered_counts_left = {l: c for l, c in lang2counts_left.items() if l in lang_set}
        filtered_counts_right = {l: c for l, c in lang2counts_right.items() if l in lang_set}
        
        output_path = os.path.join(HTML_DIR, f'mal_n_analysis_{filename}.html')
        
        stats = mal_html_report.generate_directional_mal_html_report(
            lang2MAL=filtered_lang2MAL,
            lang2counts_left=filtered_counts_left,
            lang2counts_right=filtered_counts_right,
            langNames=langNames,
            output_path=output_path,
            min_count=MIN_COUNT,
            langnameGroup=langnameGroup,
            wals_languages_path=WALS_LANGUAGES_PATH,
            title_suffix=title,
            description=description,
            lang_to_vo=lang_to_vo
        )
        
        print(f"\n✓ Generated: {stats['output_path']}")
        print(f"  - Total: {stats['n_languages']} languages")
        print(f"  - Pure Left: {stats['n_languages_left']} languages with data")
        print(f"  - Pure Right: {stats['n_languages_right']} languages with data")

## 3. MAL Dynamics Analysis

This section analyzes the **temporal dynamics** of MAL: how constituent size changes step-by-step.

### 3.1 Step-by-Step Compliance

Analyzes **which specific transitions** follow MAL for each language:

| Measure | Description |
|---------|-------------|
| **Step Compliance** | For each transition (1→2, 2→3, 3→4): is MAL_{n} > MAL_{n+1}? |
| **Compliance Category** | Fully compliant / First-step only / Partial / Anti-MAL |
| **Compliance Count** | Number of decreasing transitions (0 to 3 for n=1..4) |
| **Weighted Score** | Early transitions weighted more: w₁(1→2) + w₂(2→3) + w₃(3→4) |

**Categories**:
- 🟢 **Fully MAL-conformant**: All steps decrease (MAL₁ > MAL₂ > MAL₃ > MAL₄)
- 🟢 **First-step compliant**: At least MAL₁ > MAL₂ (most important linguistically)
- 🟡 **Partial**: Some steps decrease, but not monotonic
- 🔴 **Anti-MAL**: Sizes increase with n (MAL₁ < MAL₄)

In [ ]:
def compute_step_compliance(lang2MAL, score_type='total'):
    """
    Compute step-by-step MAL effect score for each language.
    
    Returns DataFrame with:
    - step_1_2, step_2_3, step_3_4: Boolean for each transition
    - compliance_count: Number of decreasing transitions
    - weighted_score: Weighted compliance (early steps matter more)
    - category: Compliance category
    - beta_1max, beta_3max: Log-log regression β scores
    """
    results = []
    
    # Weights for each step (earlier steps more important)
    weights = {(1, 2): 0.5, (2, 3): 0.3, (3, 4): 0.2}
    
    for lang, mal_data_all in lang2MAL.items():
        mal_data = mal_data_all.get(score_type, {})
        
        # Need at least n=1 and n=2
        if 1 not in mal_data or 2 not in mal_data:
            continue
        
        # Get MAL values for n=1,2,3,4
        mal_values = {n: mal_data.get(n) for n in range(1, 5)}
        
        # Compute step compliance (True = decreasing = MAL compliant)
        steps = {}
        for (n1, n2), weight in weights.items():
            if mal_values[n1] is not None and mal_values[n2] is not None:
                steps[(n1, n2)] = mal_values[n1] > mal_values[n2]  # True if decreasing
            else:
                steps[(n1, n2)] = None
        
        # Count compliant steps
        valid_steps = [v for v in steps.values() if v is not None]
        compliance_count = sum(v for v in valid_steps if v)
        total_steps = len(valid_steps)
        
        # Weighted score (0 to 1, higher = more compliant)
        weighted_score = 0
        total_weight = 0
        for (n1, n2), weight in weights.items():
            if steps[(n1, n2)] is not None:
                weighted_score += weight * (1 if steps[(n1, n2)] else 0)
                total_weight += weight
        weighted_score = weighted_score / total_weight if total_weight > 0 else np.nan
        
        # Determine category
        step_1_2 = steps.get((1, 2))
        step_2_3 = steps.get((2, 3))
        step_3_4 = steps.get((3, 4))
        
        if all(v == True for v in valid_steps) and len(valid_steps) >= 2:
            category = 'Fully MAL-conformant'
        elif step_1_2 == True:
            category = 'First-step compliant'
        elif compliance_count > 0:
            category = 'Partial'
        elif mal_values[1] is not None and mal_values.get(max(k for k in mal_values if mal_values[k] is not None)) is not None:
            # Check if overall anti-MAL
            max_n = max(k for k in mal_values if mal_values[k] is not None)
            if mal_values[1] < mal_values[max_n]:
                category = 'Anti-MAL'
            else:
                category = 'Partial'
        else:
            category = 'Insufficient data'
        
        # Compute β scores using log-log regression
        reg_1max = compute_loglog_regression(mal_data, start_n=1)
        reg_3max = compute_loglog_regression(mal_data, start_n=3)
        
        beta_1max = reg_1max['beta'] if reg_1max else np.nan
        beta_3max = reg_3max['beta'] if reg_3max else np.nan
        r2_1max = reg_1max['r_squared'] if reg_1max else np.nan
        r2_3max = reg_3max['r_squared'] if reg_3max else np.nan
        
        results.append({
            'language_code': lang,
            'MAL_1': mal_values[1],
            'MAL_2': mal_values[2],
            'MAL_3': mal_values[3],
            'MAL_4': mal_values[4],
            'step_1_2': step_1_2,
            'step_2_3': step_2_3,
            'step_3_4': step_3_4,
            'compliance_count': compliance_count,
            'total_steps': total_steps,
            'weighted_score': weighted_score,
            'category': category,
            # Add β scores
            'beta_1max': beta_1max,
            'beta_3max': beta_3max,
            'r2_1max': r2_1max,
            'r2_3max': r2_3max,
        })
    
    df = pd.DataFrame(results)
    if len(df) > 0:
        df['language_name'] = df['language_code'].map(lambda x: langNames.get(x, x))
        df['group'] = df['language_name'].map(lambda x: langnameGroup.get(x, 'Other'))
    
    return df


# Compute step compliance
step_compliance_df = compute_step_compliance(lang2MAL, score_type='total')

print(f"Step-by-step MAL effect score computed for {len(step_compliance_df)} languages")
print(f"\n{'='*70}")
print("COMPLIANCE CATEGORY BREAKDOWN")
print(f"{'='*70}")

category_counts = step_compliance_df['category'].value_counts()
total = len(step_compliance_df)
for cat, count in category_counts.items():
    pct = 100 * count / total
    emoji = {'Fully MAL-conformant': '🟢', 'First-step compliant': '🔵', 
             'Partial': '🟡', 'Anti-MAL': '🔴', 'Insufficient data': '⚪'}.get(cat, '')
    print(f"  {emoji} {cat}: {count} languages ({pct:.1f}%)")

print(f"\n{'='*70}")
print("β SCORES BY CATEGORY (Log-Log Regression MAL Effect)")
print(f"{'='*70}")
for cat in category_counts.index:
    cat_data = step_compliance_df[step_compliance_df['category'] == cat]
    b1 = cat_data['beta_1max'].mean()
    b3 = cat_data['beta_3max'].mean()
    print(f"  {cat}: β_1max = {b1:.3f}, β_3max = {b3:.3f}")

print(f"\n{'='*70}")
print("STEP-BY-STEP COMPLIANCE RATES")
print(f"{'='*70}")
for step in ['step_1_2', 'step_2_3', 'step_3_4']:
    valid = step_compliance_df[step].notna()
    if valid.sum() > 0:
        compliant = step_compliance_df.loc[valid, step].sum()
        pct = 100 * compliant / valid.sum()
        step_label = step.replace('_', '→').replace('step', 'MAL')
        print(f"  {step_label}: {compliant}/{valid.sum()} compliant ({pct:.1f}%)")

print(f"\n{'='*70}")
print("TOP 10 LANGUAGES BY β_1max (Strongest MAL Effect)")
print(f"{'='*70}")
top10_step = step_compliance_df.nlargest(10, 'beta_1max')[
    ['language_name', 'group', 'beta_1max', 'r2_1max', 'beta_3max', 'category']
]
print(top10_step.to_string(index=False))

#### Understanding the MAL Values Heatmap

**What are MAL values?**

MAL_n represents the **average constituent size** (in words) when a verb has exactly **n total dependents**. It's computed as the geometric mean of all constituent sizes across all verb configurations with n dependents.

- **MAL_1** = avg. size when verb has 1 dependent (VX or XV)
- **MAL_2** = avg. size when verb has 2 dependents (VXX, XVX, XXV)
- **MAL_3** = avg. size when verb has 3 dependents (VXXX, XVXX, XXVX, XXXV)
- **MAL_4** = avg. size when verb has 4 dependents (VXXXX, ..., XXXXV)

**What is "dependent count" (n)?**

The **total number of dependents** of a verb, regardless of whether they appear on the left or right side. This is the sum of left + right dependents.

**How to interpret the heatmap:**

| Reading | Interpretation |
|---------|----------------|
| **Color intensity** | Darker red = larger constituent sizes; lighter yellow = smaller sizes |
| **↓ arrows** | Size **decreased** from previous n → **MAL-conformant** transition |
| **↑ arrows** | Size **increased** from previous n → **anti-MAL** transition |
| **Row order** | Languages sorted by MAL effect score (most compliant at top) |
| **Category labels** | Right side shows compliance category (Fully, First, Partial, Anti) |

**Example reading:** If a row shows `2.50 → ↓1.80 → ↓1.45 → ↓1.20`, this language is **fully MAL-conformant** because constituent sizes consistently decrease as the number of dependents increases (2.50 > 1.80 > 1.45 > 1.20).

In [ ]:
# Visualization 1: MAL Values Heatmap with Compliance Indicators
# Shows actual MAL_n values (n=1,2,3,4) as heatmap with arrows indicating decrease/increase

# Filter to languages with at least 2 MAL values
plot_df = step_compliance_df[step_compliance_df['total_steps'] >= 2].copy()
plot_df = plot_df.sort_values('weighted_score', ascending=False)

# Limit for readability
if len(plot_df) > 60:
    print(f"Showing top 60 languages (of {len(plot_df)}) by weighted score")
    plot_df = plot_df.head(60)

# Create matrix of actual MAL values
mal_cols = ['MAL_1', 'MAL_2', 'MAL_3', 'MAL_4']
heatmap_data = plot_df.set_index('language_name')[mal_cols].copy()

# Create annotation matrix with values AND arrows showing direction
annot_matrix = heatmap_data.copy().astype(str)
for idx in heatmap_data.index:
    for i, col in enumerate(mal_cols):
        val = heatmap_data.loc[idx, col]
        if pd.notna(val):
            # Add arrow indicating direction from previous
            if i > 0:
                prev_val = heatmap_data.loc[idx, mal_cols[i-1]]
                if pd.notna(prev_val):
                    if val < prev_val:
                        arrow = "↓"  # Decreasing (MAL compliant)
                    elif val > prev_val:
                        arrow = "↑"  # Increasing (anti-MAL)
                    else:
                        arrow = "="
                    annot_matrix.loc[idx, col] = f"{arrow}{val:.2f}"
                else:
                    annot_matrix.loc[idx, col] = f"{val:.2f}"
            else:
                annot_matrix.loc[idx, col] = f"{val:.2f}"
        else:
            annot_matrix.loc[idx, col] = ""

# Add category column for context
plot_df_indexed = plot_df.set_index('language_name')
categories = plot_df_indexed.loc[heatmap_data.index, 'category']

fig, ax = plt.subplots(figsize=(12, max(14, len(plot_df) * 0.28)))

# Use colormap where lower values (smaller constituents) are cooler colors
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=annot_matrix, fmt='',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Constituent Size (MAL_n)'},
            annot_kws={'fontsize': 8})

ax.set_xticklabels(['n=1', 'n=2', 'n=3', 'n=4'], fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
ax.set_xlabel("Number of Total Dependents (n)", fontsize=12)
ax.set_ylabel("Language", fontsize=10)
ax.set_title(f"MAL Values by Dependent Count (Top {len(plot_df)} Most Compliant Languages)\n(↓ = size decreased = MAL compliant, ↑ = size increased)", fontsize=13)

# Add category labels on the right
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(np.arange(len(heatmap_data)) + 0.5)
cat_labels = [categories.iloc[i][:6] if pd.notna(categories.iloc[i]) else '' 
              for i in range(len(categories))]
ax2.set_yticklabels(cat_labels, fontsize=7, alpha=0.7)
ax2.tick_params(right=False)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_step_compliance_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {PLOTS_DIR}/mal_step_compliance_heatmap.png")

In [ ]:
# Visualization 1b: MAL Values Heatmap for ALL languages (in batches of 60)
# Creates multiple heatmaps to show the full distribution from most to least compliant

def plot_mal_heatmap_batch(batch_df, batch_num, total_batches, start_rank):
    """Plot a single batch of languages as a heatmap."""
    mal_cols = ['MAL_1', 'MAL_2', 'MAL_3', 'MAL_4']
    heatmap_data = batch_df.set_index('language_name')[mal_cols].copy()
    
    # Create annotation matrix with arrows
    annot_matrix = heatmap_data.copy().astype(str)
    for idx in heatmap_data.index:
        for i, col in enumerate(mal_cols):
            val = heatmap_data.loc[idx, col]
            if pd.notna(val):
                if i > 0:
                    prev_val = heatmap_data.loc[idx, mal_cols[i-1]]
                    if pd.notna(prev_val):
                        if val < prev_val:
                            arrow = "↓"
                        elif val > prev_val:
                            arrow = "↑"
                        else:
                            arrow = "="
                        annot_matrix.loc[idx, col] = f"{arrow}{val:.2f}"
                    else:
                        annot_matrix.loc[idx, col] = f"{val:.2f}"
                else:
                    annot_matrix.loc[idx, col] = f"{val:.2f}"
            else:
                annot_matrix.loc[idx, col] = ""
    
    # Get categories
    batch_df_indexed = batch_df.set_index('language_name')
    categories = batch_df_indexed.loc[heatmap_data.index, 'category']
    
    fig, ax = plt.subplots(figsize=(12, max(10, len(batch_df) * 0.28)))
    
    sns.heatmap(heatmap_data, cmap='YlOrRd', annot=annot_matrix, fmt='',
                linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Constituent Size (MAL_n)'},
                annot_kws={'fontsize': 8})
    
    ax.set_xticklabels(['n=1', 'n=2', 'n=3', 'n=4'], fontsize=11)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    ax.set_xlabel("Number of Total Dependents (n)", fontsize=12)
    ax.set_ylabel("Language", fontsize=10)
    
    end_rank = start_rank + len(batch_df) - 1
    ax.set_title(f"MAL Values by Dependent Count\nLanguages ranked #{start_rank}–{end_rank} by compliance (Batch {batch_num}/{total_batches})\n(↓ = MAL compliant, ↑ = anti-MAL)", fontsize=12)
    
    # Category labels on right
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(np.arange(len(heatmap_data)) + 0.5)
    cat_labels = [categories.iloc[i][:6] if pd.notna(categories.iloc[i]) else '' 
                  for i in range(len(categories))]
    ax2.set_yticklabels(cat_labels, fontsize=7, alpha=0.7)
    ax2.tick_params(right=False)
    
    plt.tight_layout()
    return fig

# Get all languages sorted by compliance
all_plot_df = step_compliance_df[step_compliance_df['total_steps'] >= 2].copy()
all_plot_df = all_plot_df.sort_values('weighted_score', ascending=False).reset_index(drop=True)

print(f"Total languages to plot: {len(all_plot_df)}")

# Split into batches of 60
batch_size = 60
n_batches = (len(all_plot_df) + batch_size - 1) // batch_size

for batch_num in range(1, n_batches + 1):
    start_idx = (batch_num - 1) * batch_size
    end_idx = min(batch_num * batch_size, len(all_plot_df))
    batch_df = all_plot_df.iloc[start_idx:end_idx]
    
    fig = plot_mal_heatmap_batch(batch_df, batch_num, n_batches, start_idx + 1)
    
    filename = f'mal_heatmap_batch_{batch_num}.png'
    plt.savefig(os.path.join(PLOTS_DIR, filename), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/{filename} (languages #{start_idx+1}–{end_idx})")
    
    # Print category breakdown for this batch
    cat_counts = batch_df['category'].value_counts()
    print(f"   Categories: {dict(cat_counts)}\n")

#### Interpreting the Results


The heatmap displays languages **sorted by MAL effect score score** (highest at top). Since we're showing only the top 60 languages, these are predominantly the most MAL-conformant ones—hence most show "Fully" (fully MAL-conformant). Languages with partial or anti-MAL patterns appear further down the list and may not be visible in this truncated view.

**What does being at the top mean?**

**Yakut**, at the top of the list, is the language with the **strongest MAL effect** among those shown. Looking at its values:
- MAL_1 ≈ 1.37 → MAL_2 ≈ 1.35 → MAL_3 ≈ 1.15 → MAL_4 ≈ (lower)

This means:
- When a Yakut verb has **1 dependent**, its average constituent size is ~1.37 words
- When it has **4 dependents**, constituents shrink to share the limited space
- All transitions show ↓ (decreasing), confirming perfect MAL effect score

**Why do some languages have larger MAL_1 values?**

Languages like **Galician** (MAL_1 ≈ 5.82) or **Catalan** (MAL_1 ≈ 4.96) have larger baseline constituent sizes. This doesn't mean they're "less compliant"—they still show consistent decreases. The compliance score is **normalized** by MAL_1, so languages with different absolute sizes can be fairly compared.

**Key insight**: Nearly all languages in this sample follow MAL, with constituent sizes consistently shrinking as the number of dependents increases. This supports the universality of the Menzerath-Altmann Law across diverse language families.

In [ ]:
# Visualization 2: Compliance Category Distribution (Pie + Bar by Family)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Pie chart of categories
category_order = ['Fully MAL-conformant', 'First-step compliant', 'Partial', 'Anti-MAL', 'Insufficient data']
category_colors = {'Fully MAL-conformant': '#2ecc71', 'First-step compliant': '#3498db', 
                   'Partial': '#f1c40f', 'Anti-MAL': '#e74c3c', 'Insufficient data': '#95a5a6'}

cat_counts = step_compliance_df['category'].value_counts()
labels = [c for c in category_order if c in cat_counts.index]
sizes = [cat_counts[c] for c in labels]
colors = [category_colors[c] for c in labels]

axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10})
axes[0].set_title('MAL effect score Categories\n(All Languages)', fontsize=14)

# Right: Stacked bar by family
family_cat = step_compliance_df.groupby(['group', 'category']).size().unstack(fill_value=0)
# Reorder columns
family_cat = family_cat[[c for c in category_order if c in family_cat.columns]]
# Sort families by proportion of fully compliant
family_cat['_sort'] = family_cat.get('Fully MAL-conformant', 0) / family_cat.sum(axis=1)
family_cat = family_cat.sort_values('_sort', ascending=True).drop('_sort', axis=1)

family_cat.plot(kind='barh', stacked=True, ax=axes[1], 
                color=[category_colors[c] for c in family_cat.columns], alpha=0.8)
axes[1].set_xlabel('Number of Languages', fontsize=12)
axes[1].set_ylabel('Language Family', fontsize=12)
axes[1].set_title('MAL effect score by Language Family', fontsize=14)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_compliance_categories.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {PLOTS_DIR}/mal_compliance_categories.png")

In [ ]:
# Visualization 3: β Scores vs VO Score (if available)
# This takes ~70 seconds per plot

if 'vo_score' in mal_vo_df.columns:
    # Merge step compliance with VO scores
    step_vo_df = pd.merge(
        step_compliance_df,
        mal_vo_df[['language_code', 'vo_score']],
        on='language_code',
        how='inner'
    )
    
    if len(step_vo_df) > 0:
        # Plot 1: β_1max vs VO Score (from step_compliance_df)
        plotting.plot_scatter_2d(
            df=step_vo_df,
            x_col='vo_score',
            y_col='beta_1max',
            group_col='group',
            appearance_dict=group_to_color,
            title='MAL Effect β_1max (n=1→max) vs VO Score',
            xlabel='VO Score (higher = more VO)',
            ylabel='β_1max (MAL effect from log-log regression)',
            figsize=(14, 10),
            label_col='language_name',
            with_labels=True,
            add_regression=True,
            plots_dir=PLOTS_DIR,
            filename='mal_step_beta_1max_vs_vo.png'
        )
        
        # Plot 2: β_3max vs VO Score
        step_vo_df_3max = step_vo_df.dropna(subset=['beta_3max'])
        if len(step_vo_df_3max) > 10:
            plotting.plot_scatter_2d(
                df=step_vo_df_3max,
                x_col='vo_score',
                y_col='beta_3max',
                group_col='group',
                appearance_dict=group_to_color,
                title='MAL Effect β_3max (n=3→max) vs VO Score',
                xlabel='VO Score (higher = more VO)',
                ylabel='β_3max (MAL effect from n=3 onwards)',
                figsize=(14, 10),
                label_col='language_name',
                with_labels=True,
                add_regression=True,
                plots_dir=PLOTS_DIR,
                filename='mal_step_beta_3max_vs_vo.png'
            )
        
        # Print correlations
        print(f"\n{'='*60}")
        print("Correlations with VO Score:")
        print(f"{'='*60}")
        corr_b1 = step_vo_df['beta_1max'].corr(step_vo_df['vo_score'])
        corr_b3 = step_vo_df['beta_3max'].corr(step_vo_df['vo_score'])
        corr_ws = step_vo_df['weighted_score'].corr(step_vo_df['vo_score'])
        print(f"  β_1max ~ VO Score: r = {corr_b1:.4f}")
        print(f"  β_3max ~ VO Score: r = {corr_b3:.4f}")
        print(f"  weighted_score ~ VO Score: r = {corr_ws:.4f}")
else:
    print("No VO score data available for correlation scatter")

In [ ]:
# Save step compliance data
step_compliance_df.to_csv(os.path.join(DATA_DIR, 'mal_step_compliance.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_step_compliance.csv")

# Print fully compliant languages
fully_compliant = step_compliance_df[step_compliance_df['category'] == 'Fully MAL-conformant']
print(f"\n{'='*70}")
print(f"FULLY MAL-COMPLIANT LANGUAGES ({len(fully_compliant)} total)")
print(f"{'='*70}")
if len(fully_compliant) > 0:
    for _, row in fully_compliant.sort_values('weighted_score', ascending=False).iterrows():
        mal_vals = f"MAL: {row['MAL_1']:.2f}→{row['MAL_2']:.2f}"
        if pd.notna(row['MAL_3']):
            mal_vals += f"→{row['MAL_3']:.2f}"
        if pd.notna(row['MAL_4']):
            mal_vals += f"→{row['MAL_4']:.2f}"
        print(f"  {row['language_name']} ({row['group']}): {mal_vals}")
else:
    print("  No fully compliant languages found")

In [ ]:
# Bar plots of MAL effect (β scores) by family - Two plots for β_1max and β_3max

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Get family colors
colors = [group_to_color.get(fam, 'gray') for fam in family_mal.index]

# Plot 1: β_1max by family
ax = axes[0]
bars = ax.barh(
    family_mal.index,
    family_mal['beta_1max_mean'],
    xerr=family_mal['beta_1max_std'],
    color=colors,
    alpha=0.7,
    capsize=3
)
ax.set_xlabel('β_1max (MAL effect from n=1→max)', fontsize=12)
ax.set_ylabel('Language Family', fontsize=12)
ax.set_title('β_1max by Language Family\n(positive = MAL effect score)', fontsize=14)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0.1, color='green', linestyle=':', alpha=0.5, label='Strong MAL (β>0.1)')
ax.axvline(x=-0.1, color='red', linestyle=':', alpha=0.5, label='Anti-MAL (β<-0.1)')
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

# Add count labels
for i, (idx, row) in enumerate(family_mal.iterrows()):
    ax.text(row['beta_1max_mean'] + row['beta_1max_std'] + 0.01,
            i, f"n={int(row['mal_compliance_count'])}", va='center', fontsize=8)

# Plot 2: β_3max by family
ax = axes[1]
# Need to sort by β_3max for this plot
family_mal_3max = family_mal.sort_values('beta_3max_mean', ascending=False)
colors_3max = [group_to_color.get(fam, 'gray') for fam in family_mal_3max.index]

bars = ax.barh(
    family_mal_3max.index,
    family_mal_3max['beta_3max_mean'],
    xerr=family_mal_3max['beta_3max_std'],
    color=colors_3max,
    alpha=0.7,
    capsize=3
)
ax.set_xlabel('β_3max (MAL effect from n=3→max)', fontsize=12)
ax.set_ylabel('Language Family', fontsize=12)
ax.set_title('β_3max by Language Family\n(excludes n=1,2)', fontsize=14)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0.1, color='green', linestyle=':', alpha=0.5, label='Strong MAL (β>0.1)')
ax.axvline(x=-0.1, color='red', linestyle=':', alpha=0.5, label='Anti-MAL (β<-0.1)')
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

# Add count labels
for i, (idx, row) in enumerate(family_mal_3max.iterrows()):
    ax.text(row['beta_3max_mean'] + row['beta_3max_std'] + 0.01,
            i, f"n={int(row['mal_compliance_count'])}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_beta_by_family.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {PLOTS_DIR}/mal_beta_by_family.png")

In [ ]:
# Statistical tests for MAL effect (β scores) by family

from scipy.stats import kruskal, mannwhitneyu, ttest_1samp

# Check that required variables exist
if 'mal_compliance_df' not in dir() or 'family_mal' not in dir():
    raise NameError("Please run all earlier cells first (cells 2-23) to create mal_compliance_df and family_mal")

print(f"{'='*70}")
print("STATISTICAL SIGNIFICANCE TESTS FOR β SCORES")
print(f"{'='*70}")

# Test both β_1max and β_3max
for beta_col, label in [('beta_1max', 'β_1max (n=1→max)'), ('beta_3max', 'β_3max (n=3→max)')]:
    print(f"\n{'='*70}")
    print(f"Testing {label}")
    print(f"{'='*70}")
    
    global_beta = mal_compliance_df[beta_col].dropna()
    
    # 1. Test if global β is significantly different from 0
    t_stat, p_global = ttest_1samp(global_beta, 0)
    print(f"\n1. One-sample t-test: Is global {beta_col} ≠ 0?")
    print(f"   Mean = {global_beta.mean():.4f}, t = {t_stat:.3f}, p = {p_global:.2e}")
    print(f"   → {'SIGNIFICANT' if p_global < 0.05 else 'Not significant'}: Languages {'do' if p_global < 0.05 else 'do not'} show MAL effect overall")
    
    # 2. Kruskal-Wallis test: Do families differ in β?
    groups = [group[beta_col].dropna().values for name, group in mal_compliance_df.groupby('group') if len(group[beta_col].dropna()) >= 3]
    if len(groups) >= 2:
        h_stat, p_kruskal = kruskal(*groups)
        print(f"\n2. Kruskal-Wallis test: Do language families differ in {beta_col}?")
        print(f"   H = {h_stat:.3f}, p = {p_kruskal:.4f}")
        print(f"   → {'SIGNIFICANT' if p_kruskal < 0.05 else 'Not significant'}: Families {'do' if p_kruskal < 0.05 else 'do not'} differ significantly")
    
    # 3. Test each family against the global mean
    print(f"\n3. Per-family tests: Is each family's {beta_col} significantly ≠ global mean?")
    print(f"   Global mean: {global_beta.mean():.4f}")
    print(f"   {'Family':<25} {'Mean':>8} {'n':>4} {'t-stat':>8} {'p-value':>10} {'Sig?':>6}")
    print(f"   {'-'*65}")
    
    global_mean = global_beta.mean()
    significant_families = []
    for family in family_mal.index:
        family_data = mal_compliance_df[mal_compliance_df['group'] == family][beta_col].dropna()
        if len(family_data) >= 3:
            t_stat, p_val = ttest_1samp(family_data, global_mean)
            sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else ''))
            print(f"   {family:<25} {family_data.mean():>8.4f} {len(family_data):>4} {t_stat:>8.3f} {p_val:>10.4f} {sig:>6}")
            if p_val < 0.05:
                direction = "higher" if family_data.mean() > global_mean else "lower"
                significant_families.append((family, direction, p_val))
    
    if significant_families:
        print(f"\n   Families significantly different from global mean:")
        for fam, direction, p in significant_families:
            print(f"   • {fam}: {direction} {beta_col} (p={p:.4f})")

### 3.2 Statistical Tests & Interpretation

**Reading the Bar Plot:**
- Each bar shows the **mean MAL effect score** for all languages in that family
- **Error bars** represent ±1 standard deviation within each family
- Numbers in parentheses indicate **sample size** (number of languages/treebanks)
- Higher (more positive) values = stronger MAL effect (more dependents → smaller constituent size)

**Statistical Significance:**

1. **Global MAL Effect**: The one-sample t-test determines if languages overall exhibit MAL behavior (compliance ≠ 0). A significant result confirms MAL is a genuine cross-linguistic tendency.

2. **Between-Family Differences**: The Kruskal-Wallis test (non-parametric ANOVA) tests whether families differ significantly in their MAL effect score. If significant (p < 0.05), family membership matters for MAL strength.

3. **Per-Family Deviations**: Individual t-tests identify which families deviate significantly from the global mean:
   - Families with **significantly higher** compliance show stronger head-planning effects
   - Families with **significantly lower** compliance may have other structural factors counteracting MAL

**Caveats:**
- Sample sizes vary greatly across families (some have 50+ treebanks, others only 3-5)
- Small families have less statistical power and larger confidence intervals
- Family groupings may obscure within-family diversity
- Significance stars: * p < 0.05, ** p < 0.01, *** p < 0.001

## 4. Language Family Analysis

### 4.1 MAL effect score by Family

## 5. Typological Analysis

### 5.1 MAL Asymmetry (Left vs Right)

**Question**: Does MAL effect score differ between left-side and right-side dependents? Does this correlate with head-directionality?

We compute:
- **MAL Asymmetry** = MAL_compliance_right - MAL_compliance_left
- Positive asymmetry → stronger MAL effect on right-side dependents
- Negative asymmetry → stronger MAL effect on left-side dependents

**Hypothesis**: VO (head-initial) languages may show stronger right-side MAL, while OV (head-final) languages show stronger left-side MAL.

In [ ]:
# 6.1 MAL Asymmetry Analysis: Left vs Right using β scores

# Merge left and right compliance scores (including β scores)
if len(mal_compliance_right) > 0 and len(mal_compliance_left) > 0:
    asymmetry_df = pd.merge(
        mal_compliance_right[['language_code', 'language_name', 'group', 'mal_compliance', 'beta_1max', 'beta_3max', 'spearman_r']],
        mal_compliance_left[['language_code', 'mal_compliance', 'beta_1max', 'beta_3max', 'spearman_r']],
        on='language_code',
        suffixes=('_right', '_left'),
        how='inner'
    )
    
    # Compute asymmetry measures using β scores
    asymmetry_df['beta_1max_asymmetry'] = asymmetry_df['beta_1max_right'] - asymmetry_df['beta_1max_left']
    asymmetry_df['beta_3max_asymmetry'] = asymmetry_df['beta_3max_right'] - asymmetry_df['beta_3max_left']
    asymmetry_df['mal_asymmetry'] = asymmetry_df['mal_compliance_right'] - asymmetry_df['mal_compliance_left']
    asymmetry_df['spearman_asymmetry'] = asymmetry_df['spearman_r_right'] - asymmetry_df['spearman_r_left']
    
    print(f"Computed MAL asymmetry for {len(asymmetry_df)} languages")
    print(f"\n{'='*70}")
    print("β SCORE ASYMMETRY SUMMARY (Right - Left)")
    print(f"{'='*70}")
    
    for asym_col, label in [('beta_1max_asymmetry', 'β_1max'), ('beta_3max_asymmetry', 'β_3max')]:
        asym_data = asymmetry_df[asym_col].dropna()
        print(f"\n{label} Asymmetry:")
        print(f"  Mean: {asym_data.mean():.4f}")
        print(f"  Positive (stronger right MAL): {(asym_data > 0).sum()} languages")
        print(f"  Negative (stronger left MAL): {(asym_data < 0).sum()} languages")
    
    # Top asymmetric languages by β_1max
    print(f"\n{'='*70}")
    print("Most RIGHT-biased MAL by β_1max (stronger MAL on right dependents):")
    print(f"{'='*70}")
    top_right = asymmetry_df.nlargest(5, 'beta_1max_asymmetry')[['language_name', 'group', 'beta_1max_asymmetry', 'beta_1max_right', 'beta_1max_left']]
    print(top_right.to_string(index=False))
    
    print(f"\n{'='*70}")
    print("Most LEFT-biased MAL by β_1max (stronger MAL on left dependents):")
    print(f"{'='*70}")
    top_left = asymmetry_df.nsmallest(5, 'beta_1max_asymmetry')[['language_name', 'group', 'beta_1max_asymmetry', 'beta_1max_right', 'beta_1max_left']]
    print(top_left.to_string(index=False))
else:
    print("Insufficient directional MAL data for asymmetry analysis")
    asymmetry_df = pd.DataFrame()

In [ ]:
# Visualization: β Score Asymmetry (Left vs Right) with Two Plots for β_1max and β_3max
# This takes ~50 seconds per plot

from adjustText import adjust_text

if len(asymmetry_df) > 0:
    
    # --- Plot 1a: Left vs Right β_1max scatter ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    for idx, (beta_col, title_suffix) in enumerate([('beta_1max', 'β_1max (n=1→max)'), ('beta_3max', 'β_3max (n=3→max)')]):
        ax = axes[idx]
        col_left = f'{beta_col}_left'
        col_right = f'{beta_col}_right'
        
        # Skip if columns don't exist or all NaN
        if col_left not in asymmetry_df.columns or asymmetry_df[col_left].isna().all():
            ax.text(0.5, 0.5, f'No data for {beta_col}', ha='center', va='center', transform=ax.transAxes)
            continue
        
        texts = []
        for group in asymmetry_df['group'].unique():
            subset = asymmetry_df[asymmetry_df['group'] == group].dropna(subset=[col_left, col_right])
            if len(subset) == 0:
                continue
            color = group_to_color.get(group, 'gray')
            ax.scatter(subset[col_left], subset[col_right], 
                       c=color, alpha=0.7, s=60, label=group)
            # Add language labels
            for _, row in subset.iterrows():
                texts.append(ax.text(row[col_left], row[col_right], 
                                      row['language_name'], fontsize=9, alpha=0.8))
        
        # Adjust text to avoid overlaps - pass ax parameter
        if texts:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.5))
        
        # Diagonal line (symmetric)
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=2, label='Symmetric')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(f'{beta_col} (Left dependents)', fontsize=12)
        ax.set_ylabel(f'{beta_col} (Right dependents)', fontsize=12)
        ax.set_title(f'Left vs Right {title_suffix}\n(above diagonal = stronger right MAL)', fontsize=14)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_beta_asymmetry_left_vs_right.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_beta_asymmetry_left_vs_right.png")
    
    # --- Plot 2: β_1max Asymmetry by family ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    for idx, (asym_col, title_suffix) in enumerate([('beta_1max_asymmetry', 'β_1max'), ('beta_3max_asymmetry', 'β_3max')]):
        ax = axes[idx]
        
        asym_data = asymmetry_df.dropna(subset=[asym_col])
        family_asymmetry = asym_data.groupby('group')[asym_col].agg(['mean', 'std', 'count'])
        family_asymmetry = family_asymmetry.sort_values('mean')
        
        colors = [group_to_color.get(fam, 'gray') for fam in family_asymmetry.index]
        bars = ax.barh(family_asymmetry.index, family_asymmetry['mean'], 
                       xerr=family_asymmetry['std'], color=colors, alpha=0.7, capsize=3)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
        ax.set_xlabel(f'{title_suffix} Asymmetry (+ = right-biased, - = left-biased)', fontsize=12)
        ax.set_ylabel('Language Family', fontsize=12)
        ax.set_title(f'{title_suffix} Asymmetry by Language Family', fontsize=14)
        ax.grid(axis='x', alpha=0.3)
        
        # Add count labels
        for i, (fam, row) in enumerate(family_asymmetry.iterrows()):
            ax.text(row['mean'] + row['std'] + 0.01, i, f"(n={int(row['count'])})", va='center', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_beta_asymmetry_by_family.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_beta_asymmetry_by_family.png")
    
    # --- Plot 3: β Asymmetry vs VO score (both β_1max and β_3max) ---
    if 'vo_score' in mal_vo_df.columns:
        asymmetry_vo = pd.merge(asymmetry_df, mal_vo_df[['language_code', 'vo_score']], 
                                 on='language_code', how='inner')
        if len(asymmetry_vo) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            
            for idx, (asym_col, title_suffix) in enumerate([('beta_1max_asymmetry', 'β_1max'), ('beta_3max_asymmetry', 'β_3max')]):
                ax = axes[idx]
                plot_data = asymmetry_vo.dropna(subset=[asym_col])
                
                if len(plot_data) < 5:
                    ax.text(0.5, 0.5, f'Insufficient data for {title_suffix}', ha='center', va='center', transform=ax.transAxes)
                    continue
                
                texts = []
                for group in plot_data['group'].unique():
                    subset = plot_data[plot_data['group'] == group]
                    color = group_to_color.get(group, 'gray')
                    ax.scatter(subset['vo_score'], subset[asym_col], 
                               c=color, alpha=0.7, s=60, label=group)
                    # Add language labels
                    for _, row in subset.iterrows():
                        texts.append(ax.text(row['vo_score'], row[asym_col], 
                                              row['language_name'], fontsize=9, alpha=0.8))
                
                # Adjust text to avoid overlaps - pass ax parameter
                if texts:
                    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.5))
                
                # Add regression line
                slope, intercept, r, p, se = scipy_stats.linregress(plot_data['vo_score'], plot_data[asym_col])
                x_line = np.linspace(plot_data['vo_score'].min(), plot_data['vo_score'].max(), 100)
                ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, 
                        label=f'Regression: r={r:.3f}, p={p:.3f}')
                
                ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
                ax.set_xlabel('VO Score (higher = more VO)', fontsize=12)
                ax.set_ylabel(f'{title_suffix} Asymmetry (+ = right-biased)', fontsize=12)
                ax.set_title(f'{title_suffix} Asymmetry vs Word Order\n(r={r:.3f}, p={p:.3f})', fontsize=14)
                ax.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.02, 1))
                ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_DIR, 'mal_beta_asymmetry_vs_vo.png'), dpi=150, bbox_inches='tight')
            plt.show()
            print(f"✓ Saved: {PLOTS_DIR}/mal_beta_asymmetry_vs_vo.png")
            
            # Print correlations
            print(f"\n{'='*60}")
            print("Correlations: β Asymmetry ~ VO Score")
            print(f"{'='*60}")
            for asym_col, label in [('beta_1max_asymmetry', 'β_1max'), ('beta_3max_asymmetry', 'β_3max')]:
                data = asymmetry_vo.dropna(subset=[asym_col])
                if len(data) > 5:
                    r, p = scipy_stats.pearsonr(data['vo_score'], data[asym_col])
                    print(f"  {label} Asymmetry ~ VO Score: r={r:.4f}, p={p:.4f}")
    else:
        print("No VO score data available for asymmetry vs word order plot")

#### Interpreting the MAL Asymmetry Results

**Left Panel: Left vs Right MAL effect score Scatter**
- Each point represents a language
- **Above the diagonal**: Languages where MAL is stronger on the **right side** (right dependents shrink more as their count increases)
- **Below the diagonal**: Languages where MAL is stronger on the **left side**
- Languages near the diagonal have **symmetric MAL** across both sides

**Middle Panel: Asymmetry by Language Family**
- Bars to the **right of zero**: Families with stronger right-side MAL
- Bars to the **left of zero**: Families with stronger left-side MAL
- Error bars show within-family variation

**Right Panel: Asymmetry vs VO Score**
- Tests whether word order predicts which side shows stronger MAL
- **Hypothesis**: VO (head-initial) languages may show stronger right-side MAL because right dependents are more common
- **Positive correlation** would support this hypothesis
- **Weak/no correlation** suggests MAL asymmetry is independent of basic word order

**Key Questions Answered**:
1. Is MAL universal across both directions, or does one side dominate?
2. Do language families show consistent directional biases?
3. Does head-directionality predict MAL asymmetry?

### 5.2 Decay Rate Analysis

**Question**: Beyond binary compliance (yes/no decrease), how *steep* is the MAL curve? 

We compute:
- **Early decay rate** = (MAL_1 - MAL_2) / MAL_1 — relative drop in first step
- **Late decay rate** = (MAL_3 - MAL_4) / MAL_3 — relative drop in last step  
- **Total decay** = (MAL_1 - MAL_max) / MAL_1 — overall shrinkage

This reveals whether languages have "front-loaded" decay (big drop early, then flat) or "gradual" decay (consistent shrinkage throughout).

In [ ]:
# 4.2 MAL Decay Rate Analysis

def compute_decay_rates(step_compliance_df):
    """Compute decay rates for each transition."""
    results = []
    
    for _, row in step_compliance_df.iterrows():
        mal_1 = row['MAL_1']
        mal_2 = row['MAL_2']
        mal_3 = row['MAL_3']
        mal_4 = row['MAL_4']
        
        # Early decay (1→2)
        if pd.notna(mal_1) and pd.notna(mal_2) and mal_1 > 0:
            early_decay = (mal_1 - mal_2) / mal_1
        else:
            early_decay = np.nan
        
        # Mid decay (2→3)
        if pd.notna(mal_2) and pd.notna(mal_3) and mal_2 > 0:
            mid_decay = (mal_2 - mal_3) / mal_2
        else:
            mid_decay = np.nan
        
        # Late decay (3→4)
        if pd.notna(mal_3) and pd.notna(mal_4) and mal_3 > 0:
            late_decay = (mal_3 - mal_4) / mal_3
        else:
            late_decay = np.nan
        
        # Total decay
        max_n_val = mal_4 if pd.notna(mal_4) else (mal_3 if pd.notna(mal_3) else mal_2)
        if pd.notna(mal_1) and pd.notna(max_n_val) and mal_1 > 0:
            total_decay = (mal_1 - max_n_val) / mal_1
        else:
            total_decay = np.nan
        
        results.append({
            'language_code': row['language_code'],
            'language_name': row['language_name'],
            'group': row['group'],
            'early_decay_1_2': early_decay,
            'mid_decay_2_3': mid_decay,
            'late_decay_3_4': late_decay,
            'total_decay': total_decay,
            'category': row['category']
        })
    
    return pd.DataFrame(results)


decay_df = compute_decay_rates(step_compliance_df)

print(f"Computed decay rates for {len(decay_df)} languages")
print(f"\n{'='*70}")
print("DECAY RATE SUMMARY")
print(f"{'='*70}")

for col in ['early_decay_1_2', 'mid_decay_2_3', 'late_decay_3_4', 'total_decay']:
    valid = decay_df[col].dropna()
    if len(valid) > 0:
        print(f"\n{col}:")
        print(f"  Mean: {valid.mean():.4f}, Std: {valid.std():.4f}")
        print(f"  Positive (shrinking): {(valid > 0).sum()}/{len(valid)} ({100*(valid > 0).mean():.1f}%)")

# Identify decay patterns
decay_df['decay_pattern'] = 'Unknown'
mask_frontloaded = (decay_df['early_decay_1_2'] > decay_df['late_decay_3_4'].fillna(0) * 1.5)
mask_backloaded = (decay_df['late_decay_3_4'].fillna(0) > decay_df['early_decay_1_2'] * 1.5)
mask_gradual = ~mask_frontloaded & ~mask_backloaded & (decay_df['early_decay_1_2'] > 0) & (decay_df['late_decay_3_4'].fillna(0) > 0)

decay_df.loc[mask_frontloaded, 'decay_pattern'] = 'Front-loaded'
decay_df.loc[mask_backloaded, 'decay_pattern'] = 'Back-loaded'
decay_df.loc[mask_gradual, 'decay_pattern'] = 'Gradual'

print(f"\n{'='*70}")
print("DECAY PATTERNS")
print(f"{'='*70}")
pattern_counts = decay_df['decay_pattern'].value_counts()
for pattern, count in pattern_counts.items():
    print(f"  {pattern}: {count} languages")

print(f"\n{'='*70}")
print("STEEPEST EARLY DECAY (front-loaded MAL)")
print(f"{'='*70}")
top_early = decay_df.nlargest(5, 'early_decay_1_2')[['language_name', 'group', 'early_decay_1_2', 'late_decay_3_4', 'total_decay']]
print(top_early.to_string(index=False))

In [ ]:
# Visualization: Decay Rate Analysis

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Plot 1: Histogram of decay rates ---
ax = axes[0, 0]
for col, color, label in [('early_decay_1_2', 'blue', 'Early (1→2)'),
                           ('mid_decay_2_3', 'green', 'Mid (2→3)'),
                           ('late_decay_3_4', 'red', 'Late (3→4)')]:
    valid = decay_df[col].dropna()
    if len(valid) > 0:
        ax.hist(valid, bins=30, alpha=0.5, color=color, label=f'{label} (n={len(valid)})')

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Decay Rate (positive = shrinking)', fontsize=11)
ax.set_ylabel('Number of Languages', fontsize=11)
ax.set_title('Distribution of Decay Rates by Transition', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 2: Early vs Late decay scatter ---
ax = axes[0, 1]
valid_both = decay_df.dropna(subset=['early_decay_1_2', 'late_decay_3_4'])
if len(valid_both) > 0:
    for group in valid_both['group'].unique():
        subset = valid_both[valid_both['group'] == group]
        color = group_to_color.get(group, 'gray')
        ax.scatter(subset['early_decay_1_2'], subset['late_decay_3_4'], 
                   c=color, alpha=0.6, s=50)
    
    # Diagonal line
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Equal decay')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    
ax.set_xlabel('Early Decay (1→2)', fontsize=11)
ax.set_ylabel('Late Decay (3→4)', fontsize=11)
ax.set_title('Early vs Late Decay\n(above diagonal = back-loaded, below = front-loaded)', fontsize=12)
ax.grid(True, alpha=0.3)

# --- Plot 3: Decay rates by family ---
ax = axes[1, 0]
family_decay = decay_df.groupby('group')[['early_decay_1_2', 'mid_decay_2_3', 'late_decay_3_4']].mean()
family_decay = family_decay.sort_values('early_decay_1_2', ascending=True)

x = np.arange(len(family_decay))
width = 0.25
ax.barh(x - width, family_decay['early_decay_1_2'], width, label='Early (1→2)', color='blue', alpha=0.7)
ax.barh(x, family_decay['mid_decay_2_3'], width, label='Mid (2→3)', color='green', alpha=0.7)
ax.barh(x + width, family_decay['late_decay_3_4'], width, label='Late (3→4)', color='red', alpha=0.7)

ax.set_yticks(x)
ax.set_yticklabels(family_decay.index, fontsize=9)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Mean Decay Rate', fontsize=11)
ax.set_ylabel('Language Family', fontsize=11)
ax.set_title('Decay Rates by Family and Transition', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

# --- Plot 4: Decay pattern distribution ---
ax = axes[1, 1]
pattern_counts = decay_df['decay_pattern'].value_counts()
colors_pattern = {'Front-loaded': '#3498db', 'Gradual': '#2ecc71', 'Back-loaded': '#e74c3c', 'Unknown': '#95a5a6'}

# Filter to patterns with count > 0 consistently
valid_patterns = [p for p in colors_pattern.keys() if pattern_counts.get(p, 0) > 0]
sizes = [pattern_counts.get(p, 0) for p in valid_patterns]
colors = [colors_pattern[p] for p in valid_patterns]

if len(valid_patterns) > 0:
    ax.pie(sizes, labels=valid_patterns, colors=colors,
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
    ax.set_title('MAL Decay Patterns\n(Front-loaded = big early drop, Back-loaded = late drop)', fontsize=12)
else:
    ax.text(0.5, 0.5, 'No decay patterns computed', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_decay_rates.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {PLOTS_DIR}/mal_decay_rates.png")

#### Interpreting the Decay Rate Results

**Top-Left: Distribution of Decay Rates by Transition**
- Histograms show how decay rates are distributed across languages
- **Positive values** = constituent size decreased (MAL-conformant)
- **Negative values** = constituent size increased (anti-MAL)
- Compare the distributions: Is early decay (blue) typically larger than late decay (red)?

**Top-Right: Early vs Late Decay Scatter**
- Each point is a language
- **Below diagonal**: "Front-loaded" languages — most size reduction happens early (1→2)
- **Above diagonal**: "Back-loaded" languages — most reduction happens late (3→4)
- **Near diagonal**: "Gradual" languages — consistent decay throughout

**Bottom-Left: Decay Rates by Family**
- Grouped bars show mean decay at each transition for each family
- Families with tall blue bars but short red bars have **front-loaded MAL**
- Families with similar bar heights have **gradual MAL**

**Bottom-Right: Decay Pattern Distribution (Pie Chart)**
- Shows the proportion of languages in each decay category
- **Front-loaded**: Big drop from n=1→2, then flattens
- **Gradual**: Consistent shrinkage at each step
- **Back-loaded**: Small early drop, bigger late drop
- **Unknown**: Insufficient data or mixed patterns

**Key Insight**: If most languages are "front-loaded," it suggests that the constraint to shrink constituents is strongest when going from 1 to 2 dependents—the first "competition" for space around the verb.

### 5.3 Trajectory Clustering

**Question**: What are the different "shapes" of MAL curves across languages? Can we identify clusters or typological patterns?

We:
1. **Normalize** MAL values: MAL_n_norm = MAL_n / MAL_1 (so all curves start at 1.0)
2. **Visualize** trajectories using parallel coordinates plots
3. **Cluster** languages by trajectory shape to identify "types"

This reveals whether languages have similar MAL "signatures" across families.

In [ ]:
# 6.3 MAL Trajectory Analysis with β scores

# Prepare normalized trajectories - include β scores from step_compliance_df
trajectory_df = step_compliance_df[['language_code', 'language_name', 'group', 'category',
                                     'MAL_1', 'MAL_2', 'MAL_3', 'MAL_4',
                                     'beta_1max', 'beta_3max', 'r2_1max', 'r2_3max']].copy()

# Normalize by MAL_1
for n in [1, 2, 3, 4]:
    col = f'MAL_{n}'
    trajectory_df[f'MAL_{n}_norm'] = trajectory_df[col] / trajectory_df['MAL_1']

# Filter to languages with at least 3 data points
trajectory_df = trajectory_df.dropna(subset=['MAL_1', 'MAL_2', 'MAL_3'])

print(f"Trajectory analysis for {len(trajectory_df)} languages with MAL_1, MAL_2, MAL_3")

# K-means clustering on normalized trajectories
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Prepare features for clustering
cluster_cols = ['MAL_2_norm', 'MAL_3_norm', 'MAL_4_norm']
cluster_data = trajectory_df[cluster_cols].dropna()

if len(cluster_data) >= 10:
    # Determine optimal k using elbow method (simplified: use 4 clusters)
    n_clusters = 4
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(cluster_data)
    
    # Add cluster labels
    trajectory_df.loc[cluster_data.index, 'trajectory_cluster'] = cluster_labels
    
    # Name clusters based on characteristics
    cluster_names = {}
    for c in range(n_clusters):
        cluster_mask = trajectory_df['trajectory_cluster'] == c
        mean_mal4_norm = trajectory_df.loc[cluster_mask, 'MAL_4_norm'].mean()
        mean_mal2_norm = trajectory_df.loc[cluster_mask, 'MAL_2_norm'].mean()
        
        if mean_mal4_norm < 0.6:
            cluster_names[c] = 'Steep decline'
        elif mean_mal4_norm > 0.9:
            cluster_names[c] = 'Flat/Weak MAL'
        elif mean_mal2_norm < 0.7:
            cluster_names[c] = 'Early steep'
        else:
            cluster_names[c] = 'Gradual decline'
    
    trajectory_df['trajectory_type'] = trajectory_df['trajectory_cluster'].map(cluster_names)
    
    print(f"\n{'='*70}")
    print(f"TRAJECTORY CLUSTERS (k={n_clusters}) with β Scores")
    print(f"{'='*70}")
    for c in range(n_clusters):
        cluster_mask = trajectory_df['trajectory_cluster'] == c
        count = cluster_mask.sum()
        mean_vals = trajectory_df.loc[cluster_mask, ['MAL_1_norm', 'MAL_2_norm', 'MAL_3_norm', 'MAL_4_norm']].mean()
        mean_beta_1 = trajectory_df.loc[cluster_mask, 'beta_1max'].mean()
        mean_beta_3 = trajectory_df.loc[cluster_mask, 'beta_3max'].mean()
        print(f"\nCluster {c} - {cluster_names[c]} ({count} languages):")
        print(f"  Mean trajectory: 1.00 → {mean_vals['MAL_2_norm']:.2f} → {mean_vals['MAL_3_norm']:.2f} → {mean_vals['MAL_4_norm']:.2f}")
        print(f"  Mean β_1max: {mean_beta_1:.3f}, Mean β_3max: {mean_beta_3:.3f}")
        
        # Top families in this cluster
        fam_counts = trajectory_df.loc[cluster_mask, 'group'].value_counts().head(3)
        print(f"  Top families: {dict(fam_counts)}")
else:
    print("Not enough data for clustering")

In [ ]:
# Visualization: Parallel Coordinates Plot for MAL Trajectories

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# --- Plot 1: All trajectories colored by family ---
ax = axes[0, 0]
ns = [1, 2, 3, 4]

for group in trajectory_df['group'].unique():
    subset = trajectory_df[trajectory_df['group'] == group]
    color = group_to_color.get(group, 'gray')
    
    for _, row in subset.iterrows():
        vals = [row[f'MAL_{n}_norm'] for n in ns]
        if not any(pd.isna(vals)):
            ax.plot(ns, vals, color=color, alpha=0.3, linewidth=0.8)

# Mean trajectory
mean_traj = [trajectory_df[f'MAL_{n}_norm'].mean() for n in ns]
ax.plot(ns, mean_traj, color='black', linewidth=3, label=f'Global mean (n={len(trajectory_df)})')

ax.set_xticks(ns)
ax.set_xticklabels(['n=1', 'n=2', 'n=3', 'n=4'])
ax.set_xlabel('Number of Dependents (n)', fontsize=11)
ax.set_ylabel('Normalized MAL (MAL_n / MAL_1)', fontsize=11)
ax.set_title('MAL Trajectories (Normalized)\nColored by Language Family', fontsize=12)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.5])

# --- Plot 2: Trajectories by cluster ---
ax = axes[0, 1]
if 'trajectory_cluster' in trajectory_df.columns:
    cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
    
    for c in sorted(trajectory_df['trajectory_cluster'].dropna().unique()):
        subset = trajectory_df[trajectory_df['trajectory_cluster'] == c]
        color = cluster_colors[int(c) % len(cluster_colors)]
        cluster_name = cluster_names.get(c, f'Cluster {int(c)}')
        
        # Plot individual trajectories
        for _, row in subset.iterrows():
            vals = [row[f'MAL_{n}_norm'] for n in ns]
            if not any(pd.isna(vals)):
                ax.plot(ns, vals, color=color, alpha=0.2, linewidth=0.5)
        
        # Plot mean trajectory for cluster
        mean_vals = [subset[f'MAL_{n}_norm'].mean() for n in ns]
        ax.plot(ns, mean_vals, color=color, linewidth=3, label=f'{cluster_name} (n={len(subset)})')
    
    ax.set_xticks(ns)
    ax.set_xticklabels(['n=1', 'n=2', 'n=3', 'n=4'])
    ax.set_xlabel('Number of Dependents (n)', fontsize=11)
    ax.set_ylabel('Normalized MAL', fontsize=11)
    ax.set_title('MAL Trajectory Clusters', fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.5])
else:
    ax.text(0.5, 0.5, 'Clustering not performed', ha='center', va='center', transform=ax.transAxes)

# --- Plot 3: Mean trajectory by family ---
ax = axes[1, 0]
family_trajectories = trajectory_df.groupby('group')[[f'MAL_{n}_norm' for n in ns]].mean()

for family in family_trajectories.index:
    color = group_to_color.get(family, 'gray')
    vals = [family_trajectories.loc[family, f'MAL_{n}_norm'] for n in ns]
    ax.plot(ns, vals, color=color, linewidth=2, marker='o', markersize=6, label=family, alpha=0.8)

ax.set_xticks(ns)
ax.set_xticklabels(['n=1', 'n=2', 'n=3', 'n=4'])
ax.set_xlabel('Number of Dependents (n)', fontsize=11)
ax.set_ylabel('Normalized MAL (family mean)', fontsize=11)
ax.set_title('Mean MAL Trajectory by Language Family', fontsize=12)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 1.1])

# --- Plot 4: Cluster distribution by family ---
ax = axes[1, 1]
if 'trajectory_type' in trajectory_df.columns:
    cluster_by_family = trajectory_df.groupby(['group', 'trajectory_type']).size().unstack(fill_value=0)
    
    # Sort by most "Steep decline"
    if 'Steep decline' in cluster_by_family.columns:
        cluster_by_family['_sort'] = cluster_by_family.get('Steep decline', 0) / cluster_by_family.sum(axis=1)
        cluster_by_family = cluster_by_family.sort_values('_sort', ascending=True).drop('_sort', axis=1)
    
    # Plot stacked bar
    cluster_colors_map = {'Steep decline': '#e74c3c', 'Gradual decline': '#3498db', 
                          'Early steep': '#2ecc71', 'Flat/Weak MAL': '#95a5a6'}
    colors = [cluster_colors_map.get(c, 'gray') for c in cluster_by_family.columns]
    
    cluster_by_family.plot(kind='barh', stacked=True, ax=ax, color=colors, alpha=0.8)
    ax.set_xlabel('Number of Languages', fontsize=11)
    ax.set_ylabel('Language Family', fontsize=11)
    ax.set_title('Trajectory Type Distribution by Family', fontsize=12)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Clustering not performed', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'mal_trajectories.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {PLOTS_DIR}/mal_trajectories.png")

#### Interpreting the Trajectory Analysis Results

**Top-Left: All Normalized Trajectories by Family**
- Each thin line is a language's MAL trajectory, normalized so MAL_1 = 1.0
- Lines going **down** = constituent sizes shrink as n increases (MAL-conformant)
- Lines staying **flat** = weak or no MAL effect
- Lines going **up** = anti-MAL (rare)
- The **black line** shows the global mean trajectory

**Top-Right: Trajectory Clusters**
- K-means clustering identifies distinct trajectory "shapes"
- Each cluster's mean trajectory is shown as a thick line
- Clusters might include:
  - **Steep decline**: Drops rapidly to ~0.5-0.6 by n=4
  - **Gradual decline**: Steady decrease to ~0.7-0.8
  - **Flat/Weak MAL**: Stays near 1.0 (little shrinkage)
  - **Early steep**: Big drop at n=2, then flattens

**Bottom-Left: Mean Trajectory by Language Family**
- Each family's average trajectory
- Families with **lower endpoints** (right side) have stronger MAL
- Families with **steeper slopes** show more dramatic shrinkage
- Compare families: Do related languages behave similarly?

**Bottom-Right: Cluster Distribution by Family**
- Shows which trajectory types are common in each family
- Families dominated by one color have consistent MAL behavior
- Mixed-color families have diverse MAL patterns

**Key Questions Answered**:
1. Are there typologically distinct MAL "signatures"?
2. Do language families cluster together in trajectory space?
3. What is the typical amount of constituent shrinkage (e.g., 20%, 40%)?

In [ ]:
# Save advanced analysis data

# Save asymmetry data
if len(asymmetry_df) > 0:
    asymmetry_df.to_csv(os.path.join(DATA_DIR, 'mal_asymmetry.csv'), index=False)
    print(f"✓ Saved: {DATA_DIR}/mal_asymmetry.csv")

# Save decay rates
decay_df.to_csv(os.path.join(DATA_DIR, 'mal_decay_rates.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_decay_rates.csv")

# Save trajectory data with clusters
trajectory_df.to_csv(os.path.join(DATA_DIR, 'mal_trajectories.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_trajectories.csv")

print(f"\n{'='*70}")
print("ADVANCED ANALYSIS SUMMARY")
print(f"{'='*70}")
print(f"\n1. MAL Asymmetry (Left vs Right):")
if len(asymmetry_df) > 0:
    print(f"   - {len(asymmetry_df)} languages analyzed")
    print(f"   - Mean asymmetry: {asymmetry_df['mal_asymmetry'].mean():.4f}")
    print(f"   - Right-biased: {(asymmetry_df['mal_asymmetry'] > 0).sum()}, Left-biased: {(asymmetry_df['mal_asymmetry'] < 0).sum()}")

print(f"\n2. Decay Rates:")
print(f"   - {len(decay_df)} languages analyzed")
print(f"   - Mean early decay (1→2): {decay_df['early_decay_1_2'].mean():.4f}")
print(f"   - Mean total decay: {decay_df['total_decay'].mean():.4f}")
print(f"   - Patterns: {dict(decay_df['decay_pattern'].value_counts())}")

print(f"\n3. Trajectory Clusters:")
if 'trajectory_type' in trajectory_df.columns:
    print(f"   - {len(trajectory_df)} languages clustered")
    print(f"   - Types: {dict(trajectory_df['trajectory_type'].value_counts())}")

In [ ]:
# Save MAL data (full structure with total/right/left)
with open(os.path.join(DATA_DIR, 'lang2MAL_full.pkl'), 'wb') as f:
    pickle.dump(lang2MAL, f)
print(f"✓ Saved: {DATA_DIR}/lang2MAL_full.pkl")

# Save individual MAL types for convenience
lang2MAL_total = {lang: data['total'] for lang, data in lang2MAL.items()}
lang2MAL_right = {lang: data['right'] for lang, data in lang2MAL.items()}
lang2MAL_left = {lang: data['left'] for lang, data in lang2MAL.items()}

with open(os.path.join(DATA_DIR, 'lang2MAL.pkl'), 'wb') as f:
    pickle.dump(lang2MAL_total, f)
print(f"✓ Saved: {DATA_DIR}/lang2MAL.pkl (total/main MAL_n)")

# Save compliance scores
mal_compliance_df.to_csv(os.path.join(DATA_DIR, 'mal_compliance_scores.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_compliance_scores.csv (main MAL_n compliance)")

mal_compliance_right.to_csv(os.path.join(DATA_DIR, 'mal_compliance_right.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_compliance_right.csv")

mal_compliance_left.to_csv(os.path.join(DATA_DIR, 'mal_compliance_left.csv'), index=False)
print(f"✓ Saved: {DATA_DIR}/mal_compliance_left.csv")

# Save family aggregates
family_mal.to_csv(os.path.join(DATA_DIR, 'mal_compliance_by_family.csv'))
print(f"✓ Saved: {DATA_DIR}/mal_compliance_by_family.csv")

# Save merged data if VO scores available
if 'vo_score' in mal_vo_df.columns:
    mal_vo_df.to_csv(os.path.join(DATA_DIR, 'mal_vo_merged.csv'), index=False)
    print(f"✓ Saved: {DATA_DIR}/mal_vo_merged.csv")

## 6. Summary and Export

### 6.1 Summary Statistics

In [ ]:
print("="*70)
print("MENZERATH-ALTMANN LAW ANALYSIS SUMMARY")
print("="*70)

print(f"\nTotal languages with MAL data: {len(lang2MAL)}")
print(f"\nCompliance scores computed:")
print(f"  MAL_n (total dependents): {len(mal_compliance_df)} languages")
print(f"  MAL_right_n (directional): {len(mal_compliance_right)} languages")
print(f"  MAL_left_n (directional): {len(mal_compliance_left)} languages")

if len(mal_compliance_df) > 0:
    print(f"\n--- Global Statistics (MAL_n total) ---")
    print(f"Mean MAL effect score: {mal_compliance_df['mal_compliance'].mean():.4f} (std: {mal_compliance_df['mal_compliance'].std():.4f})")
    print(f"Mean Spearman r: {mal_compliance_df['spearman_r'].mean():.4f}")
    print(f"Mean Decrease Ratio: {mal_compliance_df['decrease_ratio'].mean():.4f}")
    
    neg_slope_pct = (mal_compliance_df['slope'] < 0).mean() * 100
    print(f"\nLanguages with negative slope (MAL-conformant): {neg_slope_pct:.1f}%")

# Compare directional scores
if len(mal_compliance_right) > 0 and len(mal_compliance_left) > 0:
    print(f"\n--- Directional Comparison ---")
    print(f"MAL_right_n mean compliance: {mal_compliance_right['mal_compliance'].mean():.4f}")
    print(f"MAL_left_n mean compliance: {mal_compliance_left['mal_compliance'].mean():.4f}")
    print(f"MAL_right_n negative slopes: {(mal_compliance_right['slope'] < 0).mean() * 100:.1f}%")
    print(f"MAL_left_n negative slopes: {(mal_compliance_left['slope'] < 0).mean() * 100:.1f}%")

if 'vo_score' in mal_vo_df.columns:
    print(f"\n--- Correlation with Word Order ---")
    print(f"MAL effect score ~ VO Score: r = {mal_vo_df['mal_compliance'].corr(mal_vo_df['vo_score']):.4f}")
    if 'head_initiality' in mal_vo_df.columns:
        print(f"MAL effect score ~ Head-Initiality: r = {mal_vo_df['mal_compliance'].corr(mal_vo_df['head_initiality']):.4f}")

if len(mal_compliance_df) > 0:
    print(f"\n--- Top 5 MAL-Compliant Languages ---")
    top5 = mal_compliance_df.nlargest(5, 'mal_compliance')[['language_name', 'group', 'mal_compliance', 'spearman_r']]
    for _, row in top5.iterrows():
        print(f"  {row['language_name']} ({row['group']}): compliance={row['mal_compliance']:.4f}, ρ={row['spearman_r']:.4f}")
    
    print(f"\n--- Bottom 5 MAL-Compliant Languages ---")
    bottom5 = mal_compliance_df.nsmallest(5, 'mal_compliance')[['language_name', 'group', 'mal_compliance', 'spearman_r']]
    for _, row in bottom5.iterrows():
        print(f"  {row['language_name']} ({row['group']}): compliance={row['mal_compliance']:.4f}, ρ={row['spearman_r']:.4f}")

print("\n" + "="*70)
print("Output files:")
print("  - data/lang2MAL_full.pkl (full: total/right/left)")
print("  - data/lang2MAL.pkl (main MAL_n by total deps)")
print("  - data/mal_compliance_scores.csv (main compliance)")
print("  - data/mal_compliance_right.csv (directional)")
print("  - data/mal_compliance_left.csv (directional)")
print("  - data/mal_compliance_by_family.csv")
if 'vo_score' in mal_vo_df.columns:
    print("  - data/mal_vo_merged.csv")
print("  - plots/mal_*.png")
print("="*70)

### 6.2 Cross-Linguistic Universality Test

**Question**: Is MAL a genuine cross-linguistic universal, or could the observed effects arise by chance?

For each language, we test whether the MAL effect is **statistically significant** using:
1. **Permutation test**: Shuffle the MAL_n values and check if observed slope is extreme
2. **One-sample t-test**: Is the slope significantly different from 0?
3. **Bootstrap confidence intervals**: 95% CI for the slope

A language shows **significant MAL** if:
- The slope is negative AND statistically significant (p < 0.05)

We report:
- % of languages with significant MAL effect
- % by language family
- Effect sizes (Cohen's d)

In [ ]:
# Cross-Linguistic Universality Test using β scores from log-log regression
from scipy.stats import ttest_1samp, spearmanr
import numpy as np

def test_mal_significance_beta(mal_values, n_permutations=1000, alpha=0.05):
    """
    Test if a language's MAL effect is statistically significant using log-log regression.
    
    Returns dict with:
    - beta_1max: β from log-log regression (n=1→max)
    - beta_3max: β from log-log regression (n=3→max)  
    - p_value: p-value from permutation test on β_1max
    - significant: whether p < alpha
    - effect_size: Cohen's d for β_1max
    """
    ns = sorted(mal_values.keys())
    if len(ns) < 3:
        return None
    
    x = np.array(ns)
    y = np.array([mal_values[n] for n in ns])
    
    # Filter to positive values
    mask = y > 0
    if mask.sum() < 2:
        return None
    x_valid = x[mask]
    y_valid = y[mask]
    
    # Log-log regression
    log_x = np.log(x_valid)
    log_y = np.log(y_valid)
    
    # Observed slope and beta
    try:
        slope, intercept = np.polyfit(log_x, log_y, 1)
        beta = -slope  # MAL effect score
    except:
        return None
    
    # β_3max if enough data
    mask_3 = x_valid >= 3
    if mask_3.sum() >= 2:
        log_x_3 = log_x[mask_3]
        log_y_3 = log_y[mask_3]
        slope_3, _ = np.polyfit(log_x_3, log_y_3, 1)
        beta_3max = -slope_3
    else:
        beta_3max = np.nan
    
    # Permutation test: shuffle y values and compute betas
    perm_betas = []
    for _ in range(n_permutations):
        y_perm = np.random.permutation(log_y)
        perm_slope, _ = np.polyfit(log_x, y_perm, 1)
        perm_betas.append(-perm_slope)
    
    # Two-tailed p-value: proportion of permutation betas as extreme as observed
    perm_betas = np.array(perm_betas)
    p_value = np.mean(np.abs(perm_betas) >= np.abs(beta))
    
    # Effect size: Cohen's d for beta
    effect_size = (beta - np.mean(perm_betas)) / np.std(perm_betas) if np.std(perm_betas) > 0 else 0
    
    return {
        'beta_1max': beta,
        'beta_3max': beta_3max,
        'p_value': p_value,
        'significant': p_value < alpha,
        'effect_size': effect_size,
        'mal_compliant': beta > 0,  # Positive β means MAL effect score
        'significant_mal': (beta > 0) and (p_value < alpha)
    }

# Run universality test for all languages
print("="*70)
print("CROSS-LINGUISTIC UNIVERSALITY TEST (using β from log-log regression)")
print("="*70)

universality_results = []
for lang, mal_data_all in lang2MAL.items():
    mal_data = mal_data_all.get('total', {})
    if len(mal_data) >= 3:
        result = test_mal_significance_beta(mal_data, n_permutations=1000)
        if result:
            lang_name = langNames.get(lang, lang)
            group = langnameGroup.get(lang_name, 'Other')
            result['language_code'] = lang
            result['language_name'] = lang_name
            result['group'] = group
            universality_results.append(result)

universality_df = pd.DataFrame(universality_results)

if len(universality_df) > 0:
    # Overall statistics
    n_total = len(universality_df)
    n_positive_beta = (universality_df['beta_1max'] > 0).sum()
    n_significant = universality_df['significant'].sum()
    n_significant_mal = universality_df['significant_mal'].sum()
    
    print(f"\n📊 OVERALL RESULTS (n={n_total} languages)")
    print(f"   Languages with β_1max > 0 (MAL direction): {n_positive_beta} ({100*n_positive_beta/n_total:.1f}%)")
    print(f"   Languages with significant β (p<0.05):     {n_significant} ({100*n_significant/n_total:.1f}%)")
    print(f"   Languages with SIGNIFICANT MAL effect:     {n_significant_mal} ({100*n_significant_mal/n_total:.1f}%)")
    
    # β statistics
    print(f"\n📏 β SCORE STATISTICS")
    print(f"   Mean β_1max: {universality_df['beta_1max'].mean():.4f}")
    print(f"   Mean β_3max: {universality_df['beta_3max'].mean():.4f}")
    print(f"   Median β_1max: {universality_df['beta_1max'].median():.4f}")
    print(f"   Mean effect size (Cohen's d): {universality_df['effect_size'].mean():.3f}")
    
    # By language family
    print(f"\n📈 SIGNIFICANT MAL BY LANGUAGE FAMILY")
    print(f"   {'Family':<25} {'n':>4} {'Sig MAL':>8} {'%':>8} {'Mean β_1max':>12}")
    print(f"   {'-'*60}")
    
    family_stats = universality_df.groupby('group').agg({
        'significant_mal': ['sum', 'count', 'mean'],
        'beta_1max': 'mean'
    }).round(3)
    family_stats.columns = ['sig_count', 'total', 'sig_pct', 'mean_beta']
    family_stats = family_stats.sort_values('sig_pct', ascending=False)
    
    for family, row in family_stats.iterrows():
        print(f"   {family:<25} {int(row['total']):>4} {int(row['sig_count']):>8} {100*row['sig_pct']:>7.1f}% {row['mean_beta']:>12.3f}")
    
    # Statistical test: is the proportion of significant MAL > chance (5%)?
    from scipy.stats import binomtest
    
    binom_result = binomtest(n_significant_mal, n_total, p=0.05, alternative='greater')
    binom_p = binom_result.pvalue
    
    print(f"\n🔬 BINOMIAL TEST: Is significant MAL > chance level (5%)?")
    print(f"   Observed: {100*n_significant_mal/n_total:.1f}%, Expected by chance: 5%")
    print(f"   Binomial test p-value: {binom_p:.2e}")
    print(f"   → {'YES, MAL is a genuine cross-linguistic tendency!' if binom_p < 0.05 else 'Cannot reject null hypothesis'}")
    
    # Save results
    universality_df.to_csv(os.path.join(DATA_DIR, 'mal_universality_test_beta.csv'), index=False)
    print(f"\n✓ Saved: {DATA_DIR}/mal_universality_test_beta.csv")
    
    # Visualization: pie chart and β distribution
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Pie 1: Significant vs not
    ax = axes[0]
    sizes = [n_significant_mal, n_total - n_significant_mal]
    labels = [f'Significant MAL\n(β>0, p<0.05, n={n_significant_mal})', f'Not significant\n(n={n_total - n_significant_mal})']
    colors = ['#2ecc71', '#e74c3c']
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
           explode=(0.05, 0), shadow=True)
    ax.set_title(f'Cross-Linguistic Universality of MAL\n(n={n_total} languages, permutation test α=0.05)')
    
    # Pie 2: Direction of effect
    ax = axes[1]
    n_sig_negative = ((universality_df['beta_1max'] < 0) & universality_df['significant']).sum()
    n_sig_positive = n_significant_mal
    n_not_sig = n_total - n_significant
    sizes = [n_sig_positive, n_sig_negative, n_not_sig]
    labels = [f'Significant MAL\n(β>0, n={n_sig_positive})', 
              f'Significant anti-MAL\n(β<0, n={n_sig_negative})',
              f'Not significant\n(n={n_not_sig})']
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title('Direction of Significant Effects')
    
    # Histogram: β_1max distribution
    ax = axes[2]
    ax.hist(universality_df['beta_1max'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='β=0 (no effect)')
    ax.axvline(x=0.1, color='green', linestyle=':', linewidth=2, label='β=0.1 (strong MAL)')
    ax.axvline(x=-0.1, color='orange', linestyle=':', linewidth=2, label='β=-0.1 (anti-MAL)')
    ax.axvline(x=universality_df['beta_1max'].mean(), color='black', linestyle='-', linewidth=2, label=f"Mean={universality_df['beta_1max'].mean():.3f}")
    ax.set_xlabel('β_1max (MAL effect score)', fontsize=12)
    ax.set_ylabel('Number of Languages', fontsize=12)
    ax.set_title('Distribution of β_1max across Languages', fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_universality_test_beta.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_universality_test_beta.png")
else:
    print("Not enough languages with sufficient MAL data for universality test")

#### Interpretation: Cross-Linguistic Universality

**Results Summary (n=165 languages):**

| Category | Count | Percentage |
|----------|-------|------------|
| Negative slope (MAL direction) | 115 | 69.7% |
| Significant slope (p<0.05) | 63 | 38.2% |
| **Significant MAL effect** | 51 | **30.9%** |
| Significant anti-MAL | 12 | 7.3% |

**Key Findings:**

1. **~70% of languages show MAL direction**: The majority of languages exhibit the expected pattern where constituent size decreases as the number of dependents increases.

2. **~31% show statistically significant MAL**: About one-third of languages have a MAL effect strong enough to be statistically significant via permutation test. This is far above the 5% expected by chance (binomial test p < 0.001).

3. **Mean effect size = -0.66**: This is a *medium-to-large* effect (Cohen's d), indicating that MAL is not just statistically significant but also substantively meaningful.

4. **Anti-MAL is rare (~7%)**: Only 12 languages show significant effects in the *opposite* direction, suggesting MAL violations are uncommon.

**Conclusion**: MAL is a genuine cross-linguistic tendency. While not universal in the strict sense (not all languages show significant effects), the pattern is far more prevalent than chance would predict, with 70% showing the expected direction and 31% reaching statistical significance.

**Pie Chart Interpretation:**
- **Left pie**: Shows the split between languages with significant MAL (green, ~31%) vs. not significant (red, ~69%)
- **Right pie**: Breaks down by direction—green for significant MAL, red for significant anti-MAL, gray for non-significant

In [ ]:
# Plot MAL_n curves (total dependents)
lang2MAL_total = {lang: data['total'] for lang, data in lang2MAL.items() if data['total']}

fig, ax = plt.subplots(figsize=(14, 8))

# Find the maximum n value available across all languages (capped at 6)
all_n_values = set()
for data in lang2MAL_total.values():
    all_n_values.update(data.keys())
max_n = min(max(all_n_values) if all_n_values else 4, 6)  # Cap at 6
ns = list(range(1, max_n + 1))

# Filter languages that have at least n=1 data
filtered_data = {
    lang: data for lang, data in lang2MAL_total.items()
    if 1 in data  # Only require n=1
}
print(f"Plotting {len(filtered_data)} languages with MAL_n data (n=1 to {max_n}, variable length)")

# Use global MIN_COUNT defined at the top of the notebook

if filtered_data:
    # Plot individual curves (lines extend only as far as data is available)
    for lang, data in filtered_data.items():
        lang_name = langNames.get(lang, lang)
        group = langnameGroup.get(lang_name, 'Other')
        color = group_to_color.get(group, 'gray')
        # Get available n values for this language
        lang_ns = sorted([n for n in ns if n in data])
        values = [data[n] for n in lang_ns]
        ax.plot(lang_ns, values, alpha=0.3, color=color)
    
    # Plot mean curve (only where we have data)
    mean_vals = []
    mean_ns = []
    for n in ns:
        vals = [filtered_data[l][n] for l in filtered_data if n in filtered_data[l]]
        if vals:  # Only add if we have data for this n
            mean_vals.append(np.mean(vals))
            mean_ns.append(n)
    
    ax.plot(mean_ns, mean_vals, color='black', linewidth=2.5, label=f'Mean (n={len(filtered_data)})')
    print(f"Mean MAL values: {[f'n={n}: {v:.3f} (from {sum(1 for l in filtered_data if n in filtered_data[l])} langs)' for n, v in zip(mean_ns, mean_vals)]}")
    
    ax.set_xlabel("Total number of dependents (n)", fontsize=12)
    ax.set_ylabel("Average constituent size (MAL_n)", fontsize=12)
    ax.set_title(f"MAL_n: Constituent Size vs Total Dependents ({len(filtered_data)} languages)\n(minimum {MIN_COUNT} occurrences per configuration)", fontsize=12)
    ax.set_xticks(ns)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'mal_n_total_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {PLOTS_DIR}/mal_n_total_curves.png")
else:
    print("Not enough languages with MAL_n data")
    plt.close()

### 6.3 Export to HTML Report

Generate an HTML report combining all the plots and markdown text from this notebook, and add it to the index.html.

In [ ]:
# Generate HTML Report for MAL Analysis
# Iterates through notebook: copies markdown cells, detects and embeds saved plots from code cells

import os
import base64
import json
import re
from datetime import datetime

def markdown_to_html(md_text):
    """Convert basic markdown to HTML."""
    html = md_text
    
    # Convert headers
    html = re.sub(r'^######\s+(.+)$', r'<h6>\1</h6>', html, flags=re.MULTILINE)
    html = re.sub(r'^#####\s+(.+)$', r'<h5>\1</h5>', html, flags=re.MULTILINE)
    html = re.sub(r'^####\s+(.+)$', r'<h4>\1</h4>', html, flags=re.MULTILINE)
    html = re.sub(r'^###\s+(.+)$', r'<h3>\1</h3>', html, flags=re.MULTILINE)
    html = re.sub(r'^##\s+(.+)$', r'<h2>\1</h2>', html, flags=re.MULTILINE)
    html = re.sub(r'^#\s+(.+)$', r'<h1>\1</h1>', html, flags=re.MULTILINE)
    
    # Convert bold and italic
    html = re.sub(r'\*\*\*(.+?)\*\*\*', r'<strong><em>\1</em></strong>', html)
    html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', html)
    html = re.sub(r'\*(.+?)\*', r'<em>\1</em>', html)
    
    # Convert inline code
    html = re.sub(r'`([^`]+)`', r'<code>\1</code>', html)
    
    # Convert code blocks
    html = re.sub(r'```(\w*)\n(.*?)```', r'<pre><code>\2</code></pre>', html, flags=re.DOTALL)
    
    # Convert unordered lists
    lines = html.split('\n')
    in_list = False
    new_lines = []
    for line in lines:
        if re.match(r'^\s*[-*]\s+', line):
            if not in_list:
                new_lines.append('<ul>')
                in_list = True
            item = re.sub(r'^\s*[-*]\s+', '', line)
            new_lines.append(f'<li>{item}</li>')
        else:
            if in_list:
                new_lines.append('</ul>')
                in_list = False
            new_lines.append(line)
    if in_list:
        new_lines.append('</ul>')
    html = '\n'.join(new_lines)
    
    # Convert paragraphs (lines not already wrapped)
    lines = html.split('\n')
    new_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and not stripped.startswith('<'):
            new_lines.append(f'<p>{stripped}</p>')
        else:
            new_lines.append(line)
    html = '\n'.join(new_lines)
    
    # Convert LaTeX math (simple cases)
    html = re.sub(r'\$\$(.+?)\$\$', r'<div class="math">\1</div>', html, flags=re.DOTALL)
    html = re.sub(r'\$(.+?)\$', r'<span class="math">\1</span>', html)
    
    return html

def extract_plot_files_from_code(code_source):
    """
    Extract plot filenames from code cell.
    Looks for patterns like:
    - plt.savefig('filename.png')
    - plt.savefig(os.path.join(PLOTS_DIR, 'filename.png'))
    - fig.savefig(...)
    - save_path = ... followed by savefig(save_path)
    """
    plot_files = []
    
    # Pattern 1: Direct savefig with string literal
    # plt.savefig('plots/filename.png') or fig.savefig("filename.png")
    pattern1 = r'\.savefig\s*\(\s*[\'"]([^\'"]+)[\'"]'
    matches = re.findall(pattern1, code_source)
    plot_files.extend(matches)
    
    # Pattern 2: savefig with os.path.join(PLOTS_DIR, 'filename.png')
    pattern2 = r'\.savefig\s*\(\s*os\.path\.join\s*\(\s*PLOTS_DIR\s*,\s*[\'"]([^\'"]+)[\'"]'
    matches = re.findall(pattern2, code_source)
    for m in matches:
        plot_files.append(os.path.join('plots', m))
    
    # Pattern 3: savefig with os.path.join(plots_dir, 'filename.png') - lowercase variable
    pattern3 = r'\.savefig\s*\(\s*os\.path\.join\s*\(\s*plots_dir\s*,\s*[\'"]([^\'"]+)[\'"]'
    matches = re.findall(pattern3, code_source)
    for m in matches:
        plot_files.append(os.path.join('plots', m))
    
    # Pattern 4: Variable assignment then savefig - save_path = os.path.join(...)
    pattern4 = r'(\w+)\s*=\s*os\.path\.join\s*\(\s*(?:PLOTS_DIR|plots_dir)\s*,\s*[\'"]([^\'"]+)[\'"]'
    var_matches = re.findall(pattern4, code_source)
    for var_name, filename in var_matches:
        plot_files.append(os.path.join('plots', filename))
    
    # Pattern 5: plotting.py functions that take plots_dir parameter and save internally
    # Look for function calls with save_path or plots_dir parameter
    pattern5 = r'plot_\w+\s*\([^)]*plots_dir\s*=\s*PLOTS_DIR[^)]*\)'
    if re.search(pattern5, code_source):
        # Try to find the filename from the function call context
        # Look for filename patterns in the same cell
        fname_pattern = r'[\'"]([^\'\"]+\.png)[\'"]'
        fnames = re.findall(fname_pattern, code_source)
        for fn in fnames:
            if not fn.startswith('plots/'):
                plot_files.append(os.path.join('plots', fn))
            else:
                plot_files.append(fn)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_files = []
    for f in plot_files:
        if f not in seen:
            seen.add(f)
            unique_files.append(f)
    
    return unique_files

def embed_image_as_base64(image_path):
    """Read image file and return base64 encoded string."""
    if os.path.exists(image_path):
        with open(image_path, 'rb') as f:
            return base64.b64encode(f.read()).decode('utf-8')
    return None

def generate_notebook_html_report():
    """
    Generate HTML report by iterating through the notebook:
    - Markdown cells: convert to HTML and include
    - Code cells: detect savefig calls and embed the saved images
    """
    
    notebook_path = '08_menzerath_altmann_analysis.ipynb'
    html_dir = 'html_analyses'
    os.makedirs(html_dir, exist_ok=True)
    
    # Read the notebook
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)
    
    cells = notebook.get('cells', [])
    
    # Statistics
    md_count = 0
    plot_count = 0
    
    # Build HTML content
    html_content = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>MAL Notebook: Plots & Commentary</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f8f9fa;
            color: #333;
            line-height: 1.6;
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 40px;
            border-left: 4px solid #3498db;
            padding-left: 15px;
        }}
        h3 {{
            color: #555;
            margin-top: 25px;
        }}
        h4, h5, h6 {{
            color: #666;
        }}
        .plot-container {{
            background: white;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            margin: 25px 0;
            padding: 20px;
        }}
        .plot-container img {{
            max-width: 100%;
            height: auto;
            display: block;
            margin: 0 auto;
        }}
        .plot-filename {{
            font-size: 0.85em;
            color: #7f8c8d;
            text-align: center;
            margin-top: 10px;
            font-family: monospace;
        }}
        .markdown-cell {{
            background: #fff;
            border-radius: 8px;
            padding: 20px 25px;
            margin: 20px 0;
            border-left: 4px solid #27ae60;
        }}
        .markdown-cell h1, .markdown-cell h2, .markdown-cell h3, 
        .markdown-cell h4, .markdown-cell h5, .markdown-cell h6 {{
            border: none;
            padding: 0;
            margin-left: 0;
        }}
        .markdown-cell h1:first-child, .markdown-cell h2:first-child, 
        .markdown-cell h3:first-child {{
            margin-top: 0;
        }}
        .markdown-cell ul, .markdown-cell ol {{
            margin: 10px 0;
            padding-left: 25px;
        }}
        .markdown-cell li {{
            margin: 5px 0;
        }}
        .markdown-cell code {{
            background: #f4f4f4;
            padding: 2px 6px;
            border-radius: 3px;
            font-family: 'Consolas', 'Monaco', monospace;
        }}
        .markdown-cell pre {{
            background: #2d2d2d;
            color: #f8f8f2;
            padding: 15px;
            border-radius: 5px;
            overflow-x: auto;
        }}
        .markdown-cell pre code {{
            background: none;
            padding: 0;
            color: inherit;
        }}
        .math {{
            font-family: 'Times New Roman', serif;
            font-style: italic;
        }}
        .footer {{
            text-align: center;
            color: #7f8c8d;
            margin-top: 50px;
            padding-top: 20px;
            border-top: 1px solid #ddd;
        }}
        .cell-separator {{
            height: 1px;
            background: #eee;
            margin: 30px 0;
        }}
    </style>
</head>
<body>
    <h1>📓 MAL Notebook: Plots & Commentary</h1>
    <p><em>Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}</em></p>
    <p>This report mirrors the notebook structure: markdown commentary followed by any plots generated in code cells.</p>
    <hr>
'''
    
    # Iterate through all cells in order
    for i, cell in enumerate(cells):
        cell_type = cell.get('cell_type', '')
        source = ''.join(cell.get('source', []))
        
        if cell_type == 'markdown':
            # Convert markdown to HTML and add
            content = source.strip()
            if content:
                html_converted = markdown_to_html(content)
                html_content += f'''
    <div class="markdown-cell">
{html_converted}
    </div>
'''
                md_count += 1
                
        elif cell_type == 'code':
            # Check if this code cell saves any plots
            plot_files = extract_plot_files_from_code(source)
            
            for plot_file in plot_files:
                # Try to find the file
                if os.path.exists(plot_file):
                    img_data = embed_image_as_base64(plot_file)
                    if img_data:
                        filename = os.path.basename(plot_file)
                        html_content += f'''
    <div class="plot-container">
        <img src="data:image/png;base64,{img_data}" alt="{filename}">
        <div class="plot-filename">{filename}</div>
    </div>
'''
                        plot_count += 1
    
    # Add footer
    html_content += f'''
    <hr>
    <div class="footer">
        <p>Generated from notebook: 08_menzerath_altmann_analysis.ipynb</p>
        <p>Contains {md_count} markdown sections and {plot_count} plots</p>
        <p>Typometrics Project • {datetime.now().year}</p>
    </div>
</body>
</html>
'''
    
    # Write HTML file
    output_path = os.path.join(html_dir, 'mal_notebook_plots_and_commentary.html')
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✓ Generated: {output_path}")
    print(f"  - Processed {md_count} markdown cells")
    print(f"  - Embedded {plot_count} plot images")
    
    return output_path

# Generate the report
report_path = generate_notebook_html_report()

print(f"\n{'='*60}")
print("HTML REPORT GENERATED")
print(f"{'='*60}")
print(f"Report: html_analyses/mal_notebook_plots_and_commentary.html")

## 7. Generate UD Language Maps

This cell generates the `UD_maps.html` file containing two interactive maps:
1. **Map 1**: Languages by family (equal-sized dots, colored by language group)
2. **Map 2**: Languages by corpus size (dot size proportional to token count)

The info box below each map displays statistics when hovering over a language dot.

In [ ]:
# Collect UD Language Statistics
# This gathers language data from all treebanks (slow - reads CoNLL files)

import importlib
import generate_ud_maps
importlib.reload(generate_ud_maps)
from generate_ud_maps import collect_language_stats

languages_data = collect_language_stats(metadata, langNames, langnameGroup, DATA_DIR)

In [ ]:
# Generate UD Language Maps HTML
# This creates the interactive map from collected language data (fast)
importlib.reload(generate_ud_maps)

from generate_ud_maps import generate_ud_maps_html

result = generate_ud_maps_html(languages_data, HTML_DIR)